In [1]:
DATASET = '/kaggle/input/datasets/nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split/OCT2026'
MODEL = '/kaggle/input/models/varunsiddharthkaza/best-cbam-vgg16-ds/pytorch/default/1'


In [2]:
# ============================================================
# SECTION 0: DIRECTORY SETUP
# ============================================================
import os, pathlib

BASE = '/kaggle/working/davis_net_analysis'

STRUCTURE = {
    f'{BASE}/explainability/paper':       'XAI table, DRUSEN gallery (CBAM-2)',
    f'{BASE}/explainability/supp':        'CBAM-1 galleries, all-class maps',
    f'{BASE}/explainability/tables':      'ROAD/AvgDrop/CosSim CSVs',
    f'{BASE}/explainability/raw':         'Per-image localization scores',
    f'{BASE}/uncertainty/paper':          'Boxplots, class table, DRUSEN ROC',
    f'{BASE}/uncertainty/supp':           'Full distribution histograms',
    f'{BASE}/uncertainty/tables':         'Uncertainty summary CSVs',
    f'{BASE}/uncertainty/raw':            'Per-sample entropy/confidence CSV',
    f'{BASE}/calibration/paper':          'Reliability diagram, ECE table',
    f'{BASE}/calibration/tables':         'Calibration metrics CSV',
    f'{BASE}/calibration/raw':            'Val/test logits for calibration',
    f'{BASE}/conformal/paper':            'Set-size table, chi-sq, ambiguity bar',
    f'{BASE}/conformal/tables':           'Per-class conformal stats CSV',
    f'{BASE}/conformal/raw':              'Per-sample prediction sets CSV',
    f'{BASE}/joint/paper':                'Pearson r heatmap',
    f'{BASE}/joint/tables':               'Correlation stats CSV',
    f'{BASE}/joint/raw':                  'Merged per-sample joint dataframe',
    f'{BASE}/triage/paper':               'Refer rate bar chart, accuracy gap',
    f'{BASE}/triage/tables':              'Trust/Refer bucket stats CSV',
    f'{BASE}/triage/raw':                 'Per-sample triage decisions CSV',
}

for d in STRUCTURE:
    os.makedirs(d, exist_ok=True)
print(f"Created {len(STRUCTURE)} directories under {BASE}")

Created 20 directories under /kaggle/working/davis_net_analysis


In [3]:
# ============================================================
# SECTION 1: IMPORTS & GLOBAL CONFIG
# ============================================================
import random, json, glob, warnings, copy
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, pearsonr, spearmanr
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score, f1_score,
    precision_score, recall_score, brier_score_loss,
)
from sklearn.calibration import calibration_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tv_models
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Paths ────────────────────────────────────────────────────
DATASET_ROOT = '/kaggle/input/datasets/nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split/OCT2026'
TRAIN_DIR    = f'{DATASET_ROOT}/train_balanced'
VAL_DIR      = f'{DATASET_ROOT}/val_balanced'
TEST_DIR     = f'{DATASET_ROOT}/test_balanced'
MODEL_DIR    = '/kaggle/input/models/varunsiddharthkaza/best-cbam-vgg16-ds/pytorch/default/1'

# ── Constants ────────────────────────────────────────────────
CLASSES      = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
N_CLASSES    = 4
IMG_SIZE     = 224
BATCH_SIZE   = 64      
NUM_WORKERS  = 4
MC_T         = 50      # MC Dropout passes
XAI_N        = 50      # images per class for quantitative XAI (200 total)
CAL_FRAC     = 0.60    # fraction of test set used as conformal calibration

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# ── Plot Defaults ────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.1)
DPI     = 300
PALETTE = sns.color_palette('Set2', N_CLASSES)
CMAP    = dict(zip(CLASSES, PALETTE))

def savefig(path, tight=True):
    if tight: plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f"  Saved → {path}")

Device: cuda  |  GPU: Tesla T4


In [4]:
# ============================================================
# SECTION 2 (CORRECTED): DAVisNet architecture
# Key rules derived from diagnostic output:
#   block1_2  → original VGG feature indices 0–9
#   block3_4  → original VGG feature indices 10–23  (blocks 3 + 4 combined)
#   block5    → original VGG feature indices 24–30
#   cbam1/2   → CBAM(channel_attn.mlp, spatial_attn.conv)
#   aux_fc    → Sequential indices 0,1,2  (Linear,ReLU,Linear)
#   main_fc   → Sequential indices 0–7   (Linear,BN,ReLU,Linear,BN,ReLU,Dropout,Linear)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from collections import OrderedDict

class ChannelAttention(nn.Module):
    """
    Saved keys: channel_attn.mlp.0 (Linear), .2 (Linear)
    mlp index 1 is ReLU — no params, so absent from state dict.
    """
    def __init__(self, channels: int, ratio: int = 8):
        super().__init__()
        mid = channels // ratio                          # 512//8 = 64
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, mid),                    # .mlp.0
            nn.ReLU(inplace=True),                       # .mlp.1  (no saved params)
            nn.Linear(mid, channels),                    # .mlp.2
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c = x.shape[:2]
        avg = self.mlp(self.avg_pool(x).view(b, c))
        mx  = self.mlp(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg + mx).view(b, c, 1, 1)


class SpatialAttention(nn.Module):
    """
    Saved keys: spatial_attn.conv.weight [1,2,7,7], .conv.bias [1]
    """
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size,
                                 padding=kernel_size // 2)   # .conv
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    """
    Saved attribute names: channel_attn, spatial_attn
    """
    def __init__(self, channels: int, ratio: int = 8, kernel_size: int = 7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, ratio)
        self.spatial_attn = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attn(x)
        return self.spatial_attn(x)


class DAVisNet(nn.Module):
    """
    VGG16 backbone split into three blocks preserving original
    feature-layer indices as module names (critical for weight loading).

    Forward pass:
        input → block1_2 → block3_4 → cbam1 → block5 → cbam2
                                 ↓ GAP → aux_fc (aux head)
                                                         ↓ GAP → main_fc
    """
    def __init__(self, num_classes: int = 4, dropout_rate: float = 0.4):
        super().__init__()

        feats = list(tv_models.vgg16(weights=None).features.children())
        # len(feats) = 31  (indices 0–30, includes 5 MaxPool layers)

        # CRITICAL: use OrderedDict with str(original_index) as key
        # so that, e.g., the conv at position 10 in VGG is saved/loaded
        # as 'block3_4.10.weight', not 'block3_4.0.weight'.
        self.block1_2 = nn.Sequential(
            OrderedDict([(str(i), feats[i]) for i in range(0, 10)])
        )   # Conv64→Conv64→Pool→Conv128→Conv128→Pool   (indices 0–9)

        self.block3_4 = nn.Sequential(
            OrderedDict([(str(i), feats[i]) for i in range(10, 24)])
        )   # Conv256×3→Pool→Conv512×3→Pool              (indices 10–23)

        self.block5 = nn.Sequential(
            OrderedDict([(str(i), feats[i]) for i in range(24, 31)])
        )   # Conv512×3→Pool                              (indices 24–30)

        self.cbam1 = CBAM(512)     # applied after block3_4
        self.cbam2 = CBAM(512)     # applied after block5

        self.gap = nn.AdaptiveAvgPool2d(1)

        # Auxiliary head — indices 0,1,2 in Sequential
        # (saved: aux_fc.0.weight, aux_fc.2.weight; ReLU at 1 has no params)
        self.aux_fc = nn.Sequential(
            nn.Linear(512, 128),          # .0
            nn.ReLU(inplace=True),        # .1
            nn.Linear(128, num_classes),  # .2
        )

        # Main head — indices 0–7 in Sequential
        self.main_fc = nn.Sequential(
            nn.Linear(512, 512),          # .0
            nn.BatchNorm1d(512),          # .1
            nn.ReLU(inplace=True),        # .2
            nn.Linear(512, 256),          # .3
            nn.BatchNorm1d(256),          # .4
            nn.ReLU(inplace=True),        # .5
            nn.Dropout(dropout_rate),     # .6
            nn.Linear(256, num_classes),  # .7
        )

    def forward(self, x, return_aux: bool = False):
        x  = self.block1_2(x)
        x  = self.block3_4(x)
        c1 = self.cbam1(x)            # CBAM-1 output  (14×14 feature maps)
        x  = self.block5(c1)
        c2 = self.cbam2(x)            # CBAM-2 output  ( 7×7 feature maps)

        main_out = self.main_fc(self.gap(c2).flatten(1))
        if return_aux:
            aux_out = self.aux_fc(self.gap(c1).flatten(1))
            return main_out, aux_out
        return main_out

In [5]:
# ============================================================
# SECTION 3 (CORRECTED): Load weights and verify cleanly
# ============================================================
import glob, torch

MODEL_DIR = '/kaggle/input/models/varunsiddharthkaza/best-cbam-vgg16-ds/pytorch/default/1'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CLASSES = 4

def load_davis_net(model_dir: str, device: torch.device) -> nn.Module:
    pt_files = (glob.glob(f'{model_dir}/**/*.pt',  recursive=True) +
                glob.glob(f'{model_dir}/**/*.pth', recursive=True))
    if not pt_files:
        raise FileNotFoundError(f"No checkpoint found in {model_dir}")
    pt_path = sorted(pt_files)[0]
    print(f"Checkpoint: {pt_path}")

    raw = torch.load(pt_path, map_location='cpu')

    # Unwrap if nested dict
    state = raw
    if isinstance(state, dict):
        for k in ('model_state_dict', 'state_dict', 'model'):
            if k in state and isinstance(state[k], dict):
                state = state[k]
                print(f"Unwrapped from key '{k}'")
                break

    # Strip DataParallel prefix
    if any(k.startswith('module.') for k in state.keys()):
        state = {k[len('module.'):]: v for k, v in state.items()}
        print("Stripped 'module.' prefix.")

    model = DAVisNet(num_classes=N_CLASSES, dropout_rate=0.4)
    missing, unexpected = model.load_state_dict(state, strict=True)
    # strict=True intentionally — we want to KNOW if anything is wrong
    print(f"Loaded cleanly. Keys in checkpoint: {len(state)}")
    return model.to(device).eval()


model = load_davis_net(MODEL_DIR, DEVICE)

# ── logits must differ across classes ─────────────────
print("\nForward pass sanity check:")
with torch.no_grad():
    dummy = torch.randn(4, 3, 224, 224, device=DEVICE)
    out   = model(dummy)
print(f"  Output shape: {out.shape}")
print(f"  Logit std across batch (should be > 0.1): {out.std().item():.4f}")
print(f"  Logit range: [{out.min().item():.3f}, {out.max().item():.3f}]")
del dummy, out

Checkpoint: /kaggle/input/models/varunsiddharthkaza/best-cbam-vgg16-ds/pytorch/default/1/best_cbam_vgg16_ds.pt
Loaded cleanly. Keys in checkpoint: 58

Forward pass sanity check:
  Output shape: torch.Size([4, 4])
  Logit std across batch (should be > 0.1): 1.6683
  Logit range: [-3.498, 0.998]


In [6]:
# ============================================================
# SECTION 4: DATASET & DATALOADERS
# ============================================================

# Standard ImageNet normalization (matches torchvision VGG16 pretrained)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

EVAL_TF = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}


class OCTDataset(Dataset):
    def __init__(self, root: str, transform=None):
        self.transform = transform
        self.samples   = []          # (path, label_idx)
        for idx, cls in enumerate(CLASSES):
            cdir = os.path.join(root, cls)
            if not os.path.isdir(cdir):
                print(f"  WARNING: missing class dir → {cdir}")
                continue
            for p in pathlib.Path(cdir).iterdir():
                if p.suffix.lower() in EXTS:
                    self.samples.append((str(p), idx))
        counts = np.bincount([s[1] for s in self.samples], minlength=N_CLASSES)
        print(f"Dataset: {root}  |  total={len(self.samples)}"
              f"  |  {dict(zip(CLASSES, counts.tolist()))}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path     # path returned for traceability


def make_loader(root, shuffle=False, bs=BATCH_SIZE):
    ds = OCTDataset(root, transform=EVAL_TF)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      drop_last=False)


val_loader  = make_loader(VAL_DIR)
test_loader = make_loader(TEST_DIR)

Dataset: /kaggle/input/datasets/nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split/OCT2026/val_balanced  |  total=12662  |  {'CNV': 5616, 'DME': 1737, 'DRUSEN': 1327, 'NORMAL': 3982}
Dataset: /kaggle/input/datasets/nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split/OCT2026/test_balanced  |  total=12650  |  {'CNV': 5614, 'DME': 1734, 'DRUSEN': 1321, 'NORMAL': 3981}


In [7]:
# ============================================================
# SECTION 5: FULL INFERENCE PASS
# Collect logits, probs, predictions, labels, paths for
# val (temperature scaling) and test (everything else).
# ============================================================

@torch.no_grad()
def run_inference(loader, mdl, device):
    mdl.eval()
    logits_l, probs_l, preds_l, labels_l, paths_l = [], [], [], [], []
    for imgs, labels, paths in tqdm(loader, desc='  inference', leave=False):
        imgs   = imgs.to(device, non_blocking=True)
        logits = mdl(imgs)
        probs  = F.softmax(logits, dim=1)
        logits_l.append(logits.cpu()); probs_l.append(probs.cpu())
        preds_l.append(logits.argmax(1).cpu()); labels_l.append(labels)
        paths_l.extend(paths)
    return dict(
        logits = torch.cat(logits_l),   # [N, 4]  (raw, pre-softmax)
        probs  = torch.cat(probs_l),    # [N, 4]
        preds  = torch.cat(preds_l),    # [N]
        labels = torch.cat(labels_l),   # [N]
        paths  = paths_l,
    )


print("Running val inference …")
val_res  = run_inference(val_loader,  model, DEVICE)
print("Running test inference …")
test_res = run_inference(test_loader, model, DEVICE)

# ── Baseline classification report ──────────────────────────
y_true_test = test_res['labels'].numpy()
y_pred_test = test_res['preds'].numpy()
print("\n── Test Set Classification Report ──")
print(classification_report(y_true_test, y_pred_test,
                             target_names=CLASSES, digits=4))
pd.DataFrame(
    classification_report(y_true_test, y_pred_test,
                          target_names=CLASSES, digits=4, output_dict=True)
).T.to_csv(f"{BASE}/uncertainty/tables/classification_report.csv")

Running val inference …


  inference:   0%|          | 0/198 [00:00<?, ?it/s]

Running test inference …


  inference:   0%|          | 0/198 [00:00<?, ?it/s]


── Test Set Classification Report ──
              precision    recall  f1-score   support

         CNV     0.9885    0.9793    0.9839      5614
         DME     0.9561    0.9798    0.9678      1734
      DRUSEN     0.9255    0.9220    0.9238      1321
      NORMAL     0.9822    0.9857    0.9840      3981

    accuracy                         0.9754     12650
   macro avg     0.9631    0.9667    0.9649     12650
weighted avg     0.9755    0.9754    0.9754     12650



In [8]:
# ============================================================
# SECTION 6: EXPLAINABILITY
# 6.1  Hook infrastructure
# 6.2  Grad-CAM
# 6.3  Grad-CAM++
# 6.4  Score-CAM
# 6.5  Quantitative metrics (ROAD, Avg Drop, Incr Conf, Cosine Sim)
# 6.6  Figures & tables
# Target layers: model.cbam1 (CBAM-1) and model.cbam2 (CBAM-2)
# ============================================================

# ── 6.1  Hook Infrastructure ─────────────────────────────────

class HookStore:
    """Stores forward activations and backward gradients for one layer."""
    def __init__(self, module: nn.Module):
        self.act  = None
        self.grad = None
        self._fh  = module.register_forward_hook(self._fwd)
        self._bh  = module.register_full_backward_hook(self._bwd)

    def _fwd(self, m, inp, out):
        self.act = out.detach()

    def _bwd(self, m, gin, gout):
        self.grad = gout[0].detach()

    def clear(self):
        self.act = self.grad = None

    def remove(self):
        self._fh.remove(); self._bh.remove()


# ── 6.2  Grad-CAM ────────────────────────────────────────────

def gradcam(model, hook: HookStore, img_t: torch.Tensor,
            cls_idx: int, device) -> np.ndarray:
    """img_t: [1,3,H,W].  Returns heatmap [H,W] in [0,1]."""
    hook.clear()
    model.eval()
    img_v = img_t.to(device).requires_grad_(True)
    logits = model(img_v)
    model.zero_grad()
    logits[0, cls_idx].backward()

    A = hook.act[0]     # [C, h, w]
    G = hook.grad[0]    # [C, h, w]
    w = G.mean(dim=(1, 2))                          # [C]
    cam = F.relu((w.view(-1, 1, 1) * A).sum(0))    # [h, w]
    cam = (cam - cam.min()) / (cam.max() + 1e-8)
    return _upsample(cam)


# ── 6.3  Grad-CAM++ ──────────────────────────────────────────

def gradcampp(model, hook: HookStore, img_t: torch.Tensor,
              cls_idx: int, device) -> np.ndarray:
    """Uses 1st-order gradient + alpha_ij approximation (single backward pass)."""
    hook.clear()
    model.eval()
    img_v = img_t.to(device).requires_grad_(True)
    logits = model(img_v)
    model.zero_grad()
    logits[0, cls_idx].backward()

    A = hook.act[0]    # [C, h, w]
    G = hook.grad[0]   # [C, h, w]

    G2 = G ** 2
    G3 = G ** 3
    denom = 2.0 * G2 + (A * G3).sum(dim=(1, 2), keepdim=True)
    alpha = G2 / (denom + 1e-8)                              # [C, h, w]
    w = (alpha * F.relu(G)).mean(dim=(1, 2))                 # [C]
    cam = F.relu((w.view(-1, 1, 1) * A).sum(0))
    cam = (cam - cam.min()) / (cam.max() + 1e-8)
    return _upsample(cam)


# ── 6.4  Score-CAM ───────────────────────────────────────────
@torch.no_grad()
def scorecam(model, target_layer: nn.Module, img_t: torch.Tensor,
             cls_idx: int, device, inner_bs: int = 32) -> np.ndarray:
    """
    Gradient-free Score-CAM.
    img_t: [1, 3, H, W] on CPU.
    """
    model.eval()

    # Step 1: capture activations at target layer
    acts_buf = []
    h = target_layer.register_forward_hook(
        lambda m, i, o: acts_buf.append(o.detach().cpu())   # force to CPU immediately
    )
    model(img_t.to(device))
    h.remove()

    A = acts_buf[0][0]          # [C, h, w]  on CPU
    C, hs, ws = A.shape

    # Step 2: upsample and normalise each channel map (all on CPU)
    A_up = F.interpolate(
        A.unsqueeze(1),
        size=(IMG_SIZE, IMG_SIZE),
        mode='bilinear', align_corners=False
    ).squeeze(1)                # [C, H, W]  on CPU

    A_min = A_up.view(C, -1).min(1).values.view(C, 1, 1)
    A_max = A_up.view(C, -1).max(1).values.view(C, 1, 1)
    A_norm = (A_up - A_min) / (A_max - A_min + 1e-8)    # [C, H, W]  on CPU

    # Step 3: masked forward passes — build masked inputs on CPU, send batch to GPU
    img_cpu = img_t[0].cpu()    # [3, H, W]
    scores  = torch.zeros(C)

    for start in range(0, C, inner_bs):
        end   = min(start + inner_bs, C)
        masks = A_norm[start:end]                              # [bs, H, W]  CPU
        mi    = img_cpu.unsqueeze(0) * masks.unsqueeze(1)     # [bs, 3, H, W]  CPU
        out   = model(mi.to(device))                          # send to GPU here
        scores[start:end] = F.softmax(out, dim=1)[:, cls_idx].cpu()

    # Step 4: weighted sum of original (non-upsampled) activations — all CPU
    cam = F.relu((scores.view(-1, 1, 1) * A).sum(0))          # [h, w]
    cam = (cam - cam.min()) / (cam.max() + 1e-8)
    return _upsample(cam)


def _upsample(cam_tensor: torch.Tensor) -> np.ndarray:
    """Upsample [h, w] tensor to [IMG_SIZE, IMG_SIZE] numpy array."""
    return F.interpolate(
        cam_tensor.unsqueeze(0).unsqueeze(0).float(),
        size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False
    ).squeeze().cpu().numpy()


# ── 6.5  Quantitative Metrics ────────────────────────────────

def get_unnorm(img_tensor: torch.Tensor) -> np.ndarray:
    """Reverse ImageNet normalisation for display / masking (returns [H,W,3] uint8)."""
    m = torch.tensor(MEAN).view(3, 1, 1)
    s = torch.tensor(STD).view(3, 1, 1)
    img = (img_tensor.cpu().squeeze() * s + m).clamp(0, 1)
    return (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


@torch.no_grad()
def road_avgdrop(model, img_t, cam: np.ndarray,
                 cls_idx: int, device, pct: float = 0.20):
    """
    ROAD:     confidence drop (%) when top-pct% pixels removed.
    AvgDrop:  absolute drop in target-class probability (%).
    IncrConf: 1 if masked confidence > original, else 0.
    Returns (road, avg_drop, incr_conf, orig_prob, masked_prob).
    """
    model.eval()
    thr   = np.percentile(cam, (1 - pct) * 100)
    mask  = torch.tensor(cam < thr, dtype=torch.float32)  # 1=keep, 0=remove
    img0  = img_t[0].cpu()
    fill  = img0.mean()
    mi    = (img0 * mask + fill * (1 - mask)).unsqueeze(0).to(device)

    orig_prob = F.softmax(model(img_t.to(device)), dim=1)[0, cls_idx].item()
    mask_prob = F.softmax(model(mi), dim=1)[0, cls_idx].item()

    road     = max(0.0, orig_prob - mask_prob) * 100
    avg_drop = max(0.0, orig_prob - mask_prob) / (orig_prob + 1e-8) * 100
    incr     = float(mask_prob > orig_prob)
    return road, avg_drop, incr, orig_prob, mask_prob


def cosine_sim_within_class(cams: list[np.ndarray]) -> float:
    """Mean pairwise cosine similarity for a list of flattened CAM maps."""
    if len(cams) < 2:
        return float('nan')
    vecs = np.stack([c.flatten() for c in cams])   # [N, H*W]
    norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-8
    vecs  = vecs / norms
    gram  = vecs @ vecs.T                           # [N, N]
    n = len(cams)
    # Mean of upper triangle (exclude diagonal)
    idx = np.triu_indices(n, k=1)
    return float(gram[idx].mean())


# ── Select quantitative XAI subset ───────────────────────────
rng     = np.random.default_rng(SEED)
xai_idx = []
for ci in range(N_CLASSES):
    pool = np.where(y_true_test == ci)[0]
    xai_idx.extend(rng.choice(pool, min(XAI_N, len(pool)), replace=False).tolist())
xai_idx = sorted(xai_idx)
print(f"XAI subset: {len(xai_idx)} images  ({XAI_N} per class)")

# ── Attach hooks to both CBAM layers ─────────────────────────
hook_c1 = HookStore(model.cbam1)
hook_c2 = HookStore(model.cbam2)

# ── Run quantitative metrics loop ────────────────────────────
METHODS = {
    'GradCAM':   gradcam,
    'GradCAM++': gradcampp,
}
# ScoreCAM handled separately (no gradient hooks needed)

quant_records = []    # one row per (image, method, layer)
# Store CAMs by class per method/layer for cosine similarity
cam_store = {m: {'CBAM1': {c: [] for c in CLASSES},
                 'CBAM2': {c: [] for c in CLASSES}}
             for m in list(METHODS.keys()) + ['ScoreCAM']}

test_ds_flat = test_loader.dataset   # OCTDataset

for ii, glob_idx in enumerate(tqdm(xai_idx, desc='Quantitative XAI')):
    img_t, label, path = test_ds_flat[glob_idx]
    img_t  = img_t.unsqueeze(0)     # [1, 3, H, W]
    cls_nm = CLASSES[label]

    for layer_name, (hook, layer_mod) in [
        ('CBAM1', (hook_c1, model.cbam1)),
        ('CBAM2', (hook_c2, model.cbam2)),
    ]:
        # ── Gradient-based methods ──────────────────────────
        for mname, fn in METHODS.items():
            cam = fn(model, hook, img_t, label, DEVICE)
            road, adrop, incr, op, mp = road_avgdrop(
                model, img_t, cam, label, DEVICE)
            quant_records.append(dict(
                global_idx=glob_idx, label=label, cls=cls_nm,
                method=mname, layer=layer_name,
                road=road, avg_drop=adrop, incr_conf=incr,
                orig_prob=op, masked_prob=mp,
            ))
            cam_store[mname][layer_name][cls_nm].append(cam)

        # ── Score-CAM (only on CBAM2 to limit compute; runs on CBAM1 also if you want) ──
        if layer_name == 'CBAM2':
            cam_sc = scorecam(model, layer_mod, img_t, label, DEVICE)
            road_sc, adrop_sc, incr_sc, op_sc, mp_sc = road_avgdrop(
                model, img_t, cam_sc, label, DEVICE)
            quant_records.append(dict(
                global_idx=glob_idx, label=label, cls=cls_nm,
                method='ScoreCAM', layer='CBAM2',
                road=road_sc, avg_drop=adrop_sc, incr_conf=incr_sc,
                orig_prob=op_sc, masked_prob=mp_sc,
            ))
            cam_store['ScoreCAM']['CBAM2'][cls_nm].append(cam_sc)

quant_df = pd.DataFrame(quant_records)
quant_df.to_csv(f"{BASE}/explainability/raw/quant_xai_per_image.csv", index=False)
print(f"Quantitative XAI records: {len(quant_df)}")

# ── Build ROAD/AvgDrop/CosSim summary table (PAPER) ──────────
rows = []
for mname in ['GradCAM', 'GradCAM++', 'ScoreCAM']:
    layer = 'CBAM2'
    sub   = quant_df[(quant_df.method == mname) & (quant_df.layer == layer)]
    cos_vals = []
    for cls_nm in CLASSES:
        cams = cam_store[mname][layer][cls_nm]
        cos_vals.append(cosine_sim_within_class(cams))
    row = dict(
        Method        = mname,
        ROAD          = sub['road'].mean(),
        Average_Drop  = sub['avg_drop'].mean(),
        Incr_Conf_pct = sub['incr_conf'].mean() * 100,
        Cosine_Sim    = np.nanmean(cos_vals),
    )
    rows.append(row)

xai_table = pd.DataFrame(rows)
print("\n── XAI Quantitative Summary (CBAM-2) [PAPER] ──")
print(xai_table.to_string(index=False, float_format='%.4f'))
xai_table.to_csv(f"{BASE}/explainability/tables/xai_quant_summary_cbam2.csv", index=False)

# Per-class cosine similarity
cos_rows = []
for mname in ['GradCAM', 'GradCAM++', 'ScoreCAM']:
    row = {'Method': mname}
    for cls_nm in CLASSES:
        row[cls_nm] = cosine_sim_within_class(cam_store[mname]['CBAM2'][cls_nm])
    cos_rows.append(row)
cos_df = pd.DataFrame(cos_rows)
cos_df.to_csv(f"{BASE}/explainability/tables/cosine_sim_per_class_cbam2.csv", index=False)
print("\n── Per-Class Cosine Similarity (CBAM-2) [PAPER] ──")
print(cos_df.to_string(index=False, float_format='%.4f'))

XAI subset: 200 images  (50 per class)


Quantitative XAI:   0%|          | 0/200 [00:00<?, ?it/s]

Quantitative XAI records: 1000

── XAI Quantitative Summary (CBAM-2) [PAPER] ──
   Method    ROAD  Average_Drop  Incr_Conf_pct  Cosine_Sim
  GradCAM 50.0644       54.7835         3.0000      0.7164
GradCAM++ 49.8872       54.3867         3.0000      0.7287
 ScoreCAM 50.1224       54.7051         2.5000      0.7114

── Per-Class Cosine Similarity (CBAM-2) [PAPER] ──
   Method    CNV    DME  DRUSEN  NORMAL
  GradCAM 0.6803 0.7022  0.7193  0.7637
GradCAM++ 0.7001 0.7079  0.7415  0.7652
 ScoreCAM 0.6865 0.6699  0.7271  0.7621


In [9]:
# ── 6.6  XAI Figures ─────────────────────────────────────────

def overlay_cam(img_np: np.ndarray, cam: np.ndarray, alpha: float = 0.45):
    """Blend jet heatmap over original image. img_np: [H,W,3] uint8."""
    heatmap = (plt.cm.jet(cam)[:, :, :3] * 255).astype(np.uint8)
    return (alpha * heatmap + (1 - alpha) * img_np).astype(np.uint8)


# ── DRUSEN gallery: correct vs. misclassified [PAPER] ────────
# Correctly classified DRUSEN
drusen_correct_idx = [
    i for i in range(len(y_true_test))
    if y_true_test[i] == 2 and y_pred_test[i] == 2
]
# Misclassified DRUSEN → CNV  (primary confusion)
drusen_mis_cnv = [
    i for i in range(len(y_true_test))
    if y_true_test[i] == 2 and y_pred_test[i] == 0
]
# Misclassified DRUSEN → NORMAL
drusen_mis_nor = [
    i for i in range(len(y_true_test))
    if y_true_test[i] == 2 and y_pred_test[i] == 3
]

def sample_indices(idx_list, n=3):
    return rng.choice(idx_list, min(n, len(idx_list)), replace=False).tolist()

gal_correct = sample_indices(drusen_correct_idx, 3)
gal_mis_cnv = sample_indices(drusen_mis_cnv, 3)
gal_mis_nor = sample_indices(drusen_mis_nor, 3)

def make_drusen_gallery(indices, title_prefix, fname, label_for_gradcam=2):
    """
    For each index, show: original | CBAM-1 Grad-CAM | CBAM-2 Grad-CAM
    3 images per row × 3 columns.
    """
    n  = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(10, 3.5 * n))
    if n == 1: axes = axes[np.newaxis, :]
    for row, gidx in enumerate(indices):
        img_t, label, path = test_ds_flat[gidx]
        img_t  = img_t.unsqueeze(0)
        img_np = get_unnorm(img_t)
        pred   = y_pred_test[gidx]

        cam1 = gradcam(model, hook_c1, img_t, label_for_gradcam, DEVICE)
        cam2 = gradcam(model, hook_c2, img_t, label_for_gradcam, DEVICE)

        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f"True: {CLASSES[label]}  Pred: {CLASSES[pred]}")
        axes[row, 1].imshow(overlay_cam(img_np, cam1))
        axes[row, 1].set_title('Grad-CAM CBAM-1')
        axes[row, 2].imshow(overlay_cam(img_np, cam2))
        axes[row, 2].set_title('Grad-CAM CBAM-2')
        for ax in axes[row]: ax.axis('off')

    fig.suptitle(title_prefix, fontsize=13, fontweight='bold')
    savefig(f"{BASE}/explainability/paper/{fname}.png")

make_drusen_gallery(gal_correct,   'DRUSEN — Correctly Classified',       'drusen_correct_gallery')
make_drusen_gallery(gal_mis_cnv,   'DRUSEN → CNV Misclassified',          'drusen_miscls_cnv_gallery')
make_drusen_gallery(gal_mis_nor,   'DRUSEN → NORMAL Misclassified',       'drusen_miscls_normal_gallery')

# ── CBAM-1 gallery [SUPPLEMENTARY] ───────────────────────────
# (same gallery but targeting CBAM-1 heatmaps — already included in make_drusen_gallery)
print("DRUSEN galleries saved.")

# Clean up hooks
hook_c1.remove(); hook_c2.remove()

# ── Localization score (central-ROI energy ratio) for all DRUSEN test images ──
# Used later in Section 10 (joint correlation).
# Reattach a clean hook for CBAM-2 forward-pass only
hook_c2_inf = HookStore(model.cbam2)

localization_records = []
drusen_test_idx = [i for i in range(len(y_true_test)) if y_true_test[i] == 2]

for gidx in tqdm(drusen_test_idx, desc='DRUSEN localization scores'):
    img_t, label, path = test_ds_flat[gidx]
    img_t = img_t.unsqueeze(0)
    cam   = gradcam(model, hook_c2_inf, img_t, label, DEVICE)
    # Central 50% ROI energy ratio
    H, W  = cam.shape
    r0, r1 = H // 4, 3 * H // 4
    c0, c1 = W // 4, 3 * W // 4
    roi_energy   = cam[r0:r1, c0:c1].sum()
    total_energy = cam.sum() + 1e-8
    localization_records.append(dict(
        global_idx=gidx,
        label=label,
        pred=y_pred_test[gidx],
        correct=int(y_pred_test[gidx] == label),
        loc_score=float(roi_energy / total_energy),
    ))

loc_df = pd.DataFrame(localization_records)
loc_df.to_csv(f"{BASE}/explainability/raw/drusen_localization_scores.csv", index=False)
print(f"Localization scores computed for {len(loc_df)} DRUSEN images.")

# Mann-Whitney U: correct vs. incorrect localization
loc_correct = loc_df[loc_df.correct == 1]['loc_score'].values
loc_wrong   = loc_df[loc_df.correct == 0]['loc_score'].values
mw_stat, mw_p = mannwhitneyu(loc_correct, loc_wrong, alternative='greater')
print(f"\nLocalization Mann-Whitney U (correct > wrong): p = {mw_p:.4f}")
pd.DataFrame([{'stat': mw_stat, 'p_value': mw_p, 'n_correct': len(loc_correct),
               'n_wrong': len(loc_wrong)}]).to_csv(
    f"{BASE}/explainability/tables/localization_mannwhitney.csv", index=False)

hook_c2_inf.remove()

  Saved → /kaggle/working/davis_net_analysis/explainability/paper/drusen_correct_gallery.png
  Saved → /kaggle/working/davis_net_analysis/explainability/paper/drusen_miscls_cnv_gallery.png
  Saved → /kaggle/working/davis_net_analysis/explainability/paper/drusen_miscls_normal_gallery.png
DRUSEN galleries saved.


DRUSEN localization scores:   0%|          | 0/1321 [00:00<?, ?it/s]

Localization scores computed for 1321 DRUSEN images.

Localization Mann-Whitney U (correct > wrong): p = 0.0001


In [10]:
# ============================================================
# SECTION 7: MC DROPOUT UNCERTAINTY
# T=50 stochastic forward passes.
# ============================================================

def enable_mc_dropout(mdl: nn.Module):
    """Set model to eval but re-enable all Dropout layers."""
    mdl.eval()
    for m in mdl.modules():
        if isinstance(m, nn.Dropout):
            m.train()


def run_mc_dropout(loader, mdl, device, T: int = 50):
    """
    Returns dict with:
      probs_mc  [N, T, C]  — softmax probabilities per pass
      labels    [N]
      paths     list[str]
    """
    enable_mc_dropout(mdl)
    all_probs, all_labels, all_paths = [], [], []

    for imgs, labels, paths in tqdm(loader, desc=f'  MC Dropout (T={T})', leave=False):
        imgs = imgs.to(device, non_blocking=True)
        B    = imgs.size(0)
        batch_probs = torch.zeros(B, T, N_CLASSES)

        with torch.no_grad():
            for t in range(T):
                logits = mdl(imgs)
                batch_probs[:, t, :] = F.softmax(logits, dim=1).cpu()

        all_probs.append(batch_probs)
        all_labels.append(labels)
        all_paths.extend(paths)

    mdl.eval()   # restore eval (dropout disabled)
    return dict(
        probs_mc = torch.cat(all_probs),      # [N, T, C]
        labels   = torch.cat(all_labels),
        paths    = all_paths,
    )


print("Running MC Dropout on test set …")
mc = run_mc_dropout(test_loader, model, DEVICE, T=MC_T)

# ── Compute uncertainty metrics ───────────────────────────────
probs_mc  = mc['probs_mc']              # [N, T, C]
mean_prob = probs_mc.mean(dim=1)        # [N, C]
pred_mc   = mean_prob.argmax(dim=1).numpy()

# Confidence = max of mean softmax
conf = mean_prob.max(dim=1).values.numpy()                         # [N]

# Entropy of the predictive distribution (using mean_prob)
eps = 1e-8
entropy = -(mean_prob * (mean_prob + eps).log()).sum(dim=1).numpy()  # [N]

# Variation Ratio = 1 - (mode frequency / T)
mode_counts = torch.zeros(len(mc['labels']))
for t in range(MC_T):
    preds_t = probs_mc[:, t, :].argmax(dim=1)
    for i in range(len(mc['labels'])):
        if preds_t[i] == pred_mc[i]:
            mode_counts[i] += 1
var_ratio = 1.0 - (mode_counts / MC_T).numpy()                      # [N]

# Mutual Information (epistemic uncertainty): H[E[p]] - E[H[p]]
per_pass_entropy = -(probs_mc * (probs_mc + eps).log()).sum(dim=2)   # [N, T]
mi = entropy - per_pass_entropy.mean(dim=1).numpy()                  # [N]
mi = np.clip(mi, 0, None)

# STD of predicted class probability
pred_class_probs = probs_mc[torch.arange(len(mc['labels'])), :,
                            torch.tensor(pred_mc)]    # [N, T]
std_pred = pred_class_probs.std(dim=1).numpy()        # [N]

labels_np = mc['labels'].numpy()
correct   = (pred_mc == labels_np).astype(int)

mc_df = pd.DataFrame({
    'path':         mc['paths'],
    'true_class':   labels_np,
    'true_cls_nm':  [CLASSES[i] for i in labels_np],
    'pred_class':   pred_mc,
    'correct':      correct,
    'confidence':   conf,
    'entropy':      entropy,
    'variation_ratio': var_ratio,
    'mutual_info':  mi,
    'std_pred':     std_pred,
})
mc_df.to_csv(f"{BASE}/uncertainty/raw/mc_dropout_per_sample.csv", index=False)
print(f"MC Dropout records: {len(mc_df)}")

# ── Correct vs. Incorrect summary [PAPER] ────────────────────
cv = mc_df.groupby('correct')[['confidence','entropy','variation_ratio',
                                'mutual_info','std_pred']].mean()
cv.index = ['Incorrect', 'Correct']
print("\n── Correct vs. Incorrect Uncertainty ──")
print(cv.to_string(float_format='%.4f'))
cv.to_csv(f"{BASE}/uncertainty/tables/correct_vs_incorrect.csv")

# ── Statistical significance (Mann-Whitney U) [PAPER] ────────
metrics = ['entropy', 'variation_ratio', 'mutual_info', 'std_pred', 'confidence']
sig_rows = []
for m in metrics:
    a = mc_df[mc_df.correct == 0][m].values
    b = mc_df[mc_df.correct == 1][m].values
    stat, pval = mannwhitneyu(a, b, alternative='two-sided')
    sig_rows.append({'Metric': m, 'Statistic': stat, 'P_Value': pval})
sig_df = pd.DataFrame(sig_rows)
print("\n── Statistical Significance ──")
print(sig_df.to_string(index=False))
sig_df.to_csv(f"{BASE}/uncertainty/tables/significance_tests.csv", index=False)

# ── Class-wise uncertainty [PAPER] ───────────────────────────
cw = mc_df.groupby('true_cls_nm')[['confidence','entropy',
                                    'variation_ratio','mutual_info']].mean()
cw.to_csv(f"{BASE}/uncertainty/tables/classwise_uncertainty.csv")
print("\n── Class-wise Uncertainty ──")
print(cw.to_string(float_format='%.4f'))

# ── DRUSEN-specific ROC-AUC [PAPER] ──────────────────────────
drusen_mc   = mc_df[mc_df.true_cls_nm == 'DRUSEN'].copy()
drusen_err  = (drusen_mc['correct'] == 0).astype(int)   # 1 = error
roc_rows = []
for m in ['entropy', 'variation_ratio', 'mutual_info', 'confidence']:
    scores = drusen_mc[m] if m != 'confidence' else -drusen_mc[m]
    try:
        auc = roc_auc_score(drusen_err, scores)
    except Exception:
        auc = float('nan')
    roc_rows.append({'Metric': m, 'ROC_AUC': auc})
roc_df = pd.DataFrame(roc_rows)
print("\n── DRUSEN Uncertainty ROC-AUC ──")
print(roc_df.to_string(index=False, float_format='%.4f'))
roc_df.to_csv(f"{BASE}/uncertainty/tables/drusen_roc_auc.csv", index=False)

Running MC Dropout on test set …


  MC Dropout (T=50):   0%|          | 0/198 [00:00<?, ?it/s]

MC Dropout records: 12650

── Correct vs. Incorrect Uncertainty ──
           confidence  entropy  variation_ratio  mutual_info  std_pred
Incorrect      0.7286   0.5626           0.0414       0.0062    0.0355
Correct        0.9522   0.1753           0.0005       0.0042    0.0150

── Statistical Significance ──
         Metric  Statistic       P_Value
        entropy  3558660.0 4.126846e-148
variation_ratio  2305879.0 1.145594e-297
    mutual_info  2625177.0  3.283774e-29
       std_pred  3443427.0 2.274921e-128
     confidence   266600.5 3.898112e-148

── Class-wise Uncertainty ──
             confidence  entropy  variation_ratio  mutual_info
true_cls_nm                                                   
CNV              0.9648   0.1329           0.0011       0.0030
DME              0.9679   0.1218           0.0012       0.0050
DRUSEN           0.8585   0.3724           0.0047       0.0045
NORMAL           0.9413   0.2232           0.0012       0.0055

── DRUSEN Uncertainty ROC-AUC ──


ValueError: The palette dictionary is missing keys: {'1', '0'}

In [11]:
# ── Figures ──────────────────────────────────────────────────

# [PAPER] Boxplot: correct vs. incorrect for entropy + confidence
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, ylab in zip(axes,
                          ['entropy', 'confidence'],
                          ['Predictive Entropy', 'Max Softmax Confidence']):
    plot_df = mc_df.copy()
    plot_df['correct'] = plot_df['correct'].astype(int)          # force int
    plot_df['Prediction'] = plot_df['correct'].map({0: 'Incorrect', 1: 'Correct'})
    sns.boxplot(data=plot_df, x='Prediction', y=col, ax=ax,
                palette={'Incorrect': '#E07B54', 'Correct': '#5B8DB8'},
                order=['Incorrect', 'Correct'], width=0.5)
    ax.set_xlabel(''); ax.set_ylabel(ylab)
    ax.set_title(ylab)
savefig(f"{BASE}/uncertainty/paper/boxplot_correct_vs_incorrect.png")

# [PAPER] Class-wise entropy bar chart
fig, ax = plt.subplots(figsize=(7, 4))
cw_reset = cw.reset_index()
sns.barplot(data=cw_reset, x='true_cls_nm', y='entropy', ax=ax,
            palette=CMAP, order=CLASSES)
ax.set_xlabel('Class'); ax.set_ylabel('Mean Predictive Entropy')
ax.set_title('Class-wise Mean Entropy (MC Dropout, T=50)')
savefig(f"{BASE}/uncertainty/paper/classwise_entropy_bar.png")

# [PAPER] DRUSEN entropy distribution: correct vs. incorrect
fig, ax = plt.subplots(figsize=(7, 4))
for c_val, c_lab, color in [(1, 'Correct', '#5B8DB8'), (0, 'Incorrect', '#E07B54')]:
    d = drusen_mc[drusen_mc.correct == c_val]['entropy']
    sns.kdeplot(d, ax=ax, label=c_lab, color=color, fill=True, alpha=0.35)
ax.set_xlabel('Entropy'); ax.set_ylabel('Density')
ax.set_title('DRUSEN — Entropy Distribution by Correctness')
ax.legend()
savefig(f"{BASE}/uncertainty/paper/drusen_entropy_dist.png")

# [SUPPLEMENTARY] Full distributions for all metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), metrics[:4]):
    for c_val, c_lab, color in [(1,'Correct','#5B8DB8'),(0,'Incorrect','#E07B54')]:
        d = mc_df[mc_df.correct == c_val][col]
        sns.kdeplot(d, ax=ax, label=c_lab, color=color, fill=True, alpha=0.3)
    ax.set_title(col); ax.legend()
savefig(f"{BASE}/uncertainty/supp/full_metric_distributions.png")

  Saved → /kaggle/working/davis_net_analysis/uncertainty/paper/boxplot_correct_vs_incorrect.png
  Saved → /kaggle/working/davis_net_analysis/uncertainty/paper/classwise_entropy_bar.png
  Saved → /kaggle/working/davis_net_analysis/uncertainty/paper/drusen_entropy_dist.png
  Saved → /kaggle/working/davis_net_analysis/uncertainty/supp/full_metric_distributions.png


In [12]:
# ============================================================
# SECTION 8: TEMPERATURE SCALING
# Fit T* on val set logits, evaluate on test set.
# ============================================================

class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits):
        return logits / self.temperature.clamp(min=0.01)


def ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    """Expected Calibration Error."""
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    acc  = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece_val = 0.0
    for i in range(n_bins):
        mask = (conf >= bins[i]) & (conf < bins[i + 1])
        if mask.sum() > 0:
            ece_val += mask.sum() * abs(acc[mask].mean() - conf[mask].mean())
    return ece_val / len(labels)


def compute_cal_metrics(probs: np.ndarray, labels: np.ndarray):
    conf   = probs.max(axis=1)
    pred   = probs.argmax(axis=1)
    acc    = accuracy_score(labels, pred)
    ece_v  = ece(probs, labels)
    # MCE
    bins   = np.linspace(0, 1, 16)
    bin_acc, bin_conf, bin_cnt = [], [], []
    for i in range(15):
        m = (conf >= bins[i]) & (conf < bins[i + 1])
        if m.sum():
            bin_acc.append((pred[m] == labels[m]).mean())
            bin_conf.append(conf[m].mean())
            bin_cnt.append(m.sum())
    mce  = max(abs(np.array(bin_acc) - np.array(bin_conf))) if bin_acc else float('nan')
    # Brier score (multi-class): mean over N of sum_c (p_c - 1[y==c])^2
    oh   = np.zeros_like(probs)
    oh[np.arange(len(labels)), labels] = 1.0
    brier = np.mean(((probs - oh) ** 2).sum(axis=1))
    # NLL
    nll  = -np.log(probs[np.arange(len(labels)), labels] + 1e-8).mean()
    return dict(Accuracy=acc, ECE=ece_v, MCE=mce, Brier=brier, NLL=nll)


# ── Fit on val logits ─────────────────────────────────────────
val_logits  = val_res['logits']    # [N_val, 4]
val_labels  = val_res['labels']    # [N_val]
test_logits = test_res['logits']   # [N_test, 4]
test_labels_np = y_true_test

ts_model = TemperatureScaling()
optimizer = torch.optim.LBFGS([ts_model.temperature], lr=0.05, max_iter=1000)
nll_loss  = nn.CrossEntropyLoss()

def _closure():
    optimizer.zero_grad()
    loss = nll_loss(ts_model(val_logits), val_labels)
    loss.backward()
    return loss

optimizer.step(_closure)
T_opt = ts_model.temperature.item()
print(f"\nLearned temperature T* = {T_opt:.4f}")

# ── Evaluate before/after ─────────────────────────────────────
with torch.no_grad():
    probs_before = F.softmax(test_logits, dim=1).numpy()
    probs_after  = F.softmax(ts_model(test_logits), dim=1).numpy()

before_metrics = compute_cal_metrics(probs_before, test_labels_np)
after_metrics  = compute_cal_metrics(probs_after,  test_labels_np)

cal_table = pd.DataFrame({
    'Metric':     list(before_metrics.keys()),
    'Before_TS':  list(before_metrics.values()),
    'After_TS':   list(after_metrics.values()),
})
print("\n── Calibration Metrics [PAPER] ──")
print(cal_table.to_string(index=False, float_format='%.4f'))
cal_table.to_csv(f"{BASE}/calibration/tables/calibration_metrics.csv", index=False)

# DRUSEN-specific ECE
drusen_mask = test_labels_np == 2
ece_drusen_before = ece(probs_before[drusen_mask], test_labels_np[drusen_mask])
ece_drusen_after  = ece(probs_after[drusen_mask],  test_labels_np[drusen_mask])
print(f"DRUSEN ECE — Before: {ece_drusen_before:.4f} | After: {ece_drusen_after:.4f}")

# Save calibrated probs for use in conformal section
np.save(f"{BASE}/calibration/raw/test_probs_before_ts.npy", probs_before)
np.save(f"{BASE}/calibration/raw/test_probs_after_ts.npy",  probs_after)
np.save(f"{BASE}/calibration/raw/test_labels.npy", test_labels_np)

# ── Reliability Diagram [PAPER] ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, probs, title in zip(axes,
                              [probs_before, probs_after],
                              ['Before Temperature Scaling',
                               f'After Temperature Scaling (T*={T_opt:.3f})']):
    conf_arr = probs.max(axis=1)
    acc_arr  = (probs.argmax(axis=1) == test_labels_np).astype(float)
    # Bin
    bins   = np.linspace(0, 1, 16)
    b_acc, b_conf = [], []
    for i in range(15):
        mask = (conf_arr >= bins[i]) & (conf_arr < bins[i + 1])
        if mask.sum():
            b_acc.append(acc_arr[mask].mean())
            b_conf.append(conf_arr[mask].mean())
        else:
            b_acc.append(float('nan')); b_conf.append((bins[i]+bins[i+1])/2)
    ax.bar(b_conf, b_acc, width=0.065, alpha=0.75, label='Model', color='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', label='Perfect')
    ece_v = ece(probs, test_labels_np)
    ax.set_title(f"{title}\nECE = {ece_v:.4f}")
    ax.set_xlabel('Mean Confidence'); ax.set_ylabel('Fraction Correct')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend()
savefig(f"{BASE}/calibration/paper/reliability_diagram.png")


Learned temperature T* = 0.7077

── Calibration Metrics [PAPER] ──
  Metric  Before_TS  After_TS
Accuracy     0.9754    0.9754
     ECE     0.0272    0.0044
     MCE     0.4561    0.2650
   Brier     0.0395    0.0375
     NLL     0.0909    0.0780
DRUSEN ECE — Before: 0.0617 | After: 0.0368
  Saved → /kaggle/working/davis_net_analysis/calibration/paper/reliability_diagram.png


In [13]:
# ============================================================
# SECTION 9: CONFORMAL PREDICTION
# Split conformal on test set: 60% calibration, 40% evaluation.
# Nonconformity score = 1 - p(true class | x).
# ============================================================

# Use calibrated (post-TS) probabilities
test_probs_cal = probs_after       # [N_test, 4]
test_labels_cp = test_labels_np    # [N_test]
N_test = len(test_labels_cp)

# Reproducible stratified split
rng2 = np.random.default_rng(SEED + 1)
cal_mask = np.zeros(N_test, dtype=bool)
for ci in range(N_CLASSES):
    idx_ci = np.where(test_labels_cp == ci)[0]
    n_cal  = int(len(idx_ci) * CAL_FRAC)
    chosen = rng2.choice(idx_ci, n_cal, replace=False)
    cal_mask[chosen] = True

cal_probs,  cal_labels  = test_probs_cal[cal_mask],  test_labels_cp[cal_mask]
eval_probs, eval_labels = test_probs_cal[~cal_mask], test_labels_cp[~cal_mask]
print(f"Calibration set: {cal_mask.sum()} | Evaluation set: {(~cal_mask).sum()}")
print("Cal class counts: ", np.bincount(cal_labels, minlength=N_CLASSES).tolist())

# Nonconformity scores on calibration set
nc_scores = 1.0 - cal_probs[np.arange(len(cal_labels)), cal_labels]

# Coverage sweep
targets = [0.90, 0.95, 0.97, 0.99]
conformal_results = []

for target in targets:
    alpha   = 1.0 - target
    n_cal   = len(cal_labels)
    # Quantile: ceil((n+1)(1-alpha))/n
    q_level = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
    qhat    = np.quantile(nc_scores, min(q_level, 1.0))

    # Build prediction sets on evaluation set
    sets = []
    for i in range(len(eval_labels)):
        pset = [CLASSES[c] for c in range(N_CLASSES)
                if (1.0 - eval_probs[i, c]) <= qhat]
        sets.append(pset)

    sizes    = np.array([len(s) for s in sets])
    covered  = np.array([CLASSES[eval_labels[i]] in sets[i]
                         for i in range(len(eval_labels))])
    coverage = covered.mean()

    conformal_results.append(dict(
        target=target, qhat=qhat,
        mean_size=sizes.mean(), coverage=coverage,
        n_multi=int((sizes > 1).sum()),
        pct_multi=(sizes > 1).mean() * 100,
    ))
    print(f"Target={target:.0%} | qhat={qhat:.4f} | "
          f"coverage={coverage:.4f} | mean_size={sizes.mean():.3f} | "
          f"multi={int((sizes>1).sum())}")

conf_sweep_df = pd.DataFrame(conformal_results)
conf_sweep_df.to_csv(f"{BASE}/conformal/tables/conformal_sweep.csv", index=False)

# ── Focus analysis at 99% target (qhat level that produces variation) ─────────
qhat_99 = conf_sweep_df.loc[conf_sweep_df.target == 0.99, 'qhat'].values[0]

pred_sets_99 = []
for i in range(len(eval_labels)):
    pset = [CLASSES[c] for c in range(N_CLASSES)
            if (1.0 - eval_probs[i, c]) <= qhat_99]
    pred_sets_99.append(pset)

sizes_99   = np.array([len(s) for s in pred_sets_99])
covered_99 = np.array([CLASSES[eval_labels[i]] in pred_sets_99[i]
                        for i in range(len(eval_labels))])

conf_df = pd.DataFrame({
    'true_class':  eval_labels,
    'true_cls_nm': [CLASSES[i] for i in eval_labels],
    'pred_set':    pred_sets_99,
    'set_size':    sizes_99,
    'covered':     covered_99.astype(int),
    'is_multi':    (sizes_99 > 1).astype(int),
})
conf_df.to_csv(f"{BASE}/conformal/raw/conformal_per_sample_99pct.csv", index=False)

# ── Class-wise set size & coverage table [PAPER] ─────────────
cw_conf = conf_df.groupby('true_cls_nm').agg(
    mean_set_size=('set_size', 'mean'),
    coverage=('covered', 'mean'),
    n_multi=('is_multi', 'sum'),
    n_total=('is_multi', 'count'),
).reindex(CLASSES)
cw_conf['pct_multi'] = cw_conf['n_multi'] / cw_conf['n_total'] * 100
print("\n── Conformal Class-wise Summary (99% target) [PAPER] ──")
print(cw_conf.to_string(float_format='%.4f'))
cw_conf.to_csv(f"{BASE}/conformal/tables/classwise_conformal_99pct.csv")

# ── Chi-square test [PAPER] ───────────────────────────────────
obs    = np.array([[cw_conf.loc[c,'n_multi'],
                    cw_conf.loc[c,'n_total'] - cw_conf.loc[c,'n_multi']]
                   for c in CLASSES])
chi2, chi_p, dof, expected = chi2_contingency(obs)
print(f"\nChi-square: χ²={chi2:.2f}, p={chi_p:.6e}, dof={dof}")
pd.DataFrame([{'chi2': chi2, 'p_value': chi_p, 'dof': dof}]).to_csv(
    f"{BASE}/conformal/tables/chi_square_ambiguity.csv", index=False)

# ── Ambiguity bar chart [PAPER] ──────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
pcts = [cw_conf.loc[c,'pct_multi'] for c in CLASSES]
bars = ax.bar(CLASSES, pcts, color=[CMAP[c] for c in CLASSES], width=0.5)
ax.set_ylabel('% Images with Multi-label Prediction Set')
ax.set_title(f'Conformal Ambiguity by Class (99% coverage, χ²={chi2:.1f}, p<0.001)')
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
savefig(f"{BASE}/conformal/paper/ambiguity_by_class_bar.png")

# ── Confusion pairings within ambiguous DRUSEN sets [PAPER] ──
drusen_multi = conf_df[(conf_df.true_cls_nm == 'DRUSEN') & (conf_df.is_multi == 1)]
pair_counts = drusen_multi['pred_set'].apply(
    lambda s: tuple(sorted(s))).value_counts()
print("\nDRUSEN ambiguous prediction-set pairings:")
print(pair_counts.to_string())
pair_counts.to_csv(f"{BASE}/conformal/tables/drusen_pairing_counts.csv")

Calibration set: 7588 | Evaluation set: 5062
Cal class counts:  [3368, 1040, 792, 2388]
Target=90% | qhat=0.0568 | coverage=0.9004 | mean_size=0.906 | multi=0
Target=95% | qhat=0.1718 | coverage=0.9502 | mean_size=0.960 | multi=0
Target=97% | qhat=0.3384 | coverage=0.9668 | mean_size=0.986 | multi=0
Target=99% | qhat=0.8808 | coverage=0.9923 | mean_size=1.058 | multi=293

── Conformal Class-wise Summary (99% target) [PAPER] ──
             mean_set_size  coverage  n_multi  n_total  pct_multi
true_cls_nm                                                      
CNV                 1.0321    0.9969       72     2246     3.2057
DME                 1.0259    0.9928       18      694     2.5937
DRUSEN              1.2457    0.9679      130      529    24.5747
NORMAL              1.0458    0.9937       73     1593     4.5825

Chi-square: χ²=387.07, p=1.397231e-83, dof=3
  Saved → /kaggle/working/davis_net_analysis/conformal/paper/ambiguity_by_class_bar.png

DRUSEN ambiguous prediction-set pairin

In [15]:
import os
os.makedirs(f"{BASE}/joint/supp", exist_ok=True)
savefig(f"{BASE}/joint/supp/scatter_pairs.png")

# ============================================================
# SECTION 10: JOINT XAI + UQ + CONFORMAL CORRELATION
# Merge: MC Dropout entropy / conformal set size / XAI loc score
# for DRUSEN test images only, then run correlations.
# ============================================================

# ── Align indices: all three sources use global test-set position ─
# mc_df rows are in the same order as test_loader.dataset.samples
# conf_df rows correspond to the evaluation split (indices ~cal_mask complement)

# Add global test-set index to mc_df
mc_df = mc_df.copy()
mc_df['global_idx'] = list(range(len(mc_df)))

# Add global test-set index to conf_df (maps to ~cal_mask rows)
eval_global_idx = np.where(~cal_mask)[0]   # positions in the test set
conf_df = conf_df.copy()
conf_df['global_idx'] = eval_global_idx

# ── DRUSEN-only slices ────────────────────────────────────────
mc_drusen   = mc_df[mc_df.true_cls_nm == 'DRUSEN'][['global_idx','entropy','correct']].copy()
conf_drusen = conf_df[conf_df.true_cls_nm == 'DRUSEN'][['global_idx','set_size','is_multi']].copy()
loc_drusen  = loc_df[['global_idx','loc_score','correct']].copy()

# Merge: only images present in all three (conformal eval split ∩ localization set)
joint = (mc_drusen
         .merge(conf_drusen, on='global_idx', how='inner')
         .merge(loc_drusen[['global_idx','loc_score']], on='global_idx', how='inner'))

print(f"\nJoint DRUSEN analysis — N = {len(joint)}")
joint.to_csv(f"{BASE}/joint/raw/joint_drusen_df.csv", index=False)

# ── Pearson correlations ──────────────────────────────────────
pairs = [
    ('entropy',   'set_size',  'Entropy vs. Conformal Set Size'),
    ('entropy',   'loc_score', 'Entropy vs. Localization Score'),
    ('set_size',  'loc_score', 'Conformal Set Size vs. Localization Score'),
]
corr_rows = []
for x_col, y_col, label in pairs:
    r, p_r = pearsonr(joint[x_col], joint[y_col])
    rho, p_s = spearmanr(joint[x_col], joint[y_col])
    corr_rows.append(dict(Pair=label,
                           Pearson_r=r, Pearson_p=p_r,
                           Spearman_rho=rho, Spearman_p=p_s))
    print(f"{label}: r={r:.3f} (p={p_r:.4f})  ρ={rho:.3f} (p={p_s:.4f})")

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(f"{BASE}/joint/tables/correlation_stats.csv", index=False)

# ── Entropy: multi-label vs. singleton for DRUSEN ────────────
grp_multi = joint[joint.is_multi == 1]['entropy'].values
grp_sing  = joint[joint.is_multi == 0]['entropy'].values
mw_s, mw_p = mannwhitneyu(grp_multi, grp_sing, alternative='two-sided')
print(f"\nEntropy multi-label={grp_multi.mean():.4f}  "
      f"singleton={grp_sing.mean():.4f}  p={mw_p:.4e}")
pd.DataFrame([{'multi_mean': grp_multi.mean(), 'sing_mean': grp_sing.mean(),
               'mw_stat': mw_s, 'p_value': mw_p}]).to_csv(
    f"{BASE}/joint/tables/entropy_multilabel_vs_singleton.csv", index=False)

# ── Localization: multi-label vs. singleton ───────────────────
loc_multi = joint[joint.is_multi == 1]['loc_score'].values
loc_sing  = joint[joint.is_multi == 0]['loc_score'].values
mw_loc_s, mw_loc_p = mannwhitneyu(loc_multi, loc_sing, alternative='two-sided')
print(f"Loc score multi={loc_multi.mean():.4f}  singleton={loc_sing.mean():.4f}  p={mw_loc_p:.4f}")

# ── Correlation matrix figure [PAPER] ────────────────────────
r_matrix = joint[['entropy','set_size','loc_score']].corr(method='pearson')
r_matrix.index   = ['Entropy', 'Set Size', 'Loc. Score']
r_matrix.columns = r_matrix.index

fig, ax = plt.subplots(figsize=(6, 5))
mask  = np.triu(np.ones_like(r_matrix, dtype=bool), k=1)
annot = r_matrix.applymap(lambda v: f'{v:.3f}')
sns.heatmap(r_matrix, annot=annot, fmt='', cmap='coolwarm',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('DRUSEN — Pearson r: Entropy × Conformal Set Size × Loc. Score')
savefig(f"{BASE}/joint/paper/correlation_heatmap.png")

# ── Scatter: Entropy vs. Conformal Set Size [SUPPLEMENTARY] ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
scatter_pairs = [('entropy','set_size'), ('entropy','loc_score'), ('set_size','loc_score')]
lbls  = ['Entropy', 'Conformal Set Size', 'Localization Score']
for ax, (xc, yc), (xl, yl) in zip(axes, scatter_pairs,
                                    [('Entropy','Conf. Set Size'),
                                     ('Entropy','Loc. Score'),
                                     ('Conf. Set Size','Loc. Score')]):
    ax.scatter(joint[xc], joint[yc], alpha=0.3, s=10, color='steelblue')
    r, _ = pearsonr(joint[xc], joint[yc])
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(f'r = {r:.3f}')
savefig(f"{BASE}/joint/supp/scatter_pairs.png")

  Saved → /kaggle/working/davis_net_analysis/joint/supp/scatter_pairs.png

Joint DRUSEN analysis — N = 529
Entropy vs. Conformal Set Size: r=0.783 (p=0.0000)  ρ=0.737 (p=0.0000)
Entropy vs. Localization Score: r=-0.060 (p=0.1672)  ρ=-0.059 (p=0.1732)
Conformal Set Size vs. Localization Score: r=-0.014 (p=0.7486)  ρ=-0.036 (p=0.4117)

Entropy multi-label=0.6254  singleton=0.2931  p=2.3475e-64
Loc score multi=0.6329  singleton=0.6371  p=0.4113
  Saved → /kaggle/working/davis_net_analysis/joint/paper/correlation_heatmap.png
  Saved → /kaggle/working/davis_net_analysis/joint/supp/scatter_pairs.png


In [16]:
# ============================================================
# SECTION 11: TRUST / REFER TRIAGE RULE
# Rebuild from the verified joint outputs.
# Rule: Refer if EITHER conformal set size > 1  OR
#       entropy > 75th-percentile threshold.
# (Localization score is an orthogonal signal and not used in
# the decision rule itself, but reported as a validation check.)
# ============================================================

# ── Build triage on the full test evaluation split ────────────
# We have mc_df (all test images) and conf_df (evaluation split only).
# For triage we use the evaluation split so conformal labels exist.

# Merge MC Dropout + conformal for all evaluation-split images
eval_mc   = mc_df[mc_df.global_idx.isin(eval_global_idx)].copy()
eval_mc   = eval_mc.set_index('global_idx')
conf_eval = conf_df.set_index('global_idx')

triage = eval_mc.join(conf_eval[['set_size','is_multi','covered']], how='inner')
triage = triage.reset_index()

# Entropy threshold: 75th percentile on calibration-split entropy
ent_thresh = mc_df.loc[mc_df.global_idx.isin(np.where(cal_mask)[0]),
                        'entropy'].quantile(0.75)
print(f"Entropy 75th-pct threshold (from cal split): {ent_thresh:.4f}")

# Decision: Refer = is_multi == 1 OR entropy > threshold
triage['refer'] = ((triage['is_multi'] == 1) |
                   (triage['entropy']  > ent_thresh)).astype(int)

trust = triage[triage.refer == 0]
refer = triage[triage.refer == 1]

trust_acc = accuracy_score(trust['true_class'], trust['pred_class'])
refer_acc = accuracy_score(refer['true_class'], refer['pred_class'])

print(f"\nTrust bucket: n={len(trust)} ({len(trust)/len(triage)*100:.1f}%)  "
      f"accuracy={trust_acc:.4f}")
print(f"Refer bucket: n={len(refer)} ({len(refer)/len(triage)*100:.1f}%)  "
      f"accuracy={refer_acc:.4f}")
print(f"Accuracy gap: {(trust_acc - refer_acc)*100:.2f} pp")

# ── Refer rate by class [PAPER] ───────────────────────────────
ref_rate = triage.groupby('true_cls_nm').agg(
    refer_count=('refer', 'sum'),
    total=('refer', 'count'),
).reindex(CLASSES)
ref_rate['refer_rate'] = ref_rate['refer_count'] / ref_rate['total']
print("\n── Refer Rate by Class ──")
print(ref_rate.to_string(float_format='%.4f'))
ref_rate.to_csv(f"{BASE}/triage/tables/refer_rate_by_class.csv")

triage_summary = pd.DataFrame([
    {'bucket': 'Trust', 'n': len(trust), 'pct': len(trust)/len(triage)*100,
     'accuracy': trust_acc},
    {'bucket': 'Refer', 'n': len(refer), 'pct': len(refer)/len(triage)*100,
     'accuracy': refer_acc},
])
triage_summary.to_csv(f"{BASE}/triage/tables/trust_refer_summary.csv", index=False)
triage.to_csv(f"{BASE}/triage/raw/per_sample_triage.csv", index=False)

# ── Refer rate bar chart [PAPER] ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
rates = ref_rate['refer_rate'].values * 100
bars  = ax.bar(CLASSES, rates, color=[CMAP[c] for c in CLASSES], width=0.5)
ax.set_ylabel('Refer Rate (%)')
ax.set_title('Refer Rate by Class — Trust/Refer Triage Rule')
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 105)
savefig(f"{BASE}/triage/paper/refer_rate_by_class.png")

# ── Trust vs. Refer accuracy gap [PAPER] ─────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Trust', 'Refer'], [trust_acc * 100, refer_acc * 100],
       color=['#5B8DB8', '#E07B54'], width=0.4)
ax.set_ylim(85, 102)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Trust vs. Refer Bucket Accuracy')
for i, val in enumerate([trust_acc, refer_acc]):
    ax.text(i, val * 100 + 0.3, f'{val*100:.2f}%', ha='center', fontsize=11)
savefig(f"{BASE}/triage/paper/trust_refer_accuracy_gap.png")

print("\n══ All sections complete. ══")
print(f"Results written to: {BASE}")

Entropy 75th-pct threshold (from cal split): 0.2426

Trust bucket: n=3742 (73.9%)  accuracy=0.9976
Refer bucket: n=1320 (26.1%)  accuracy=0.9083
Accuracy gap: 8.93 pp

── Refer Rate by Class ──
             refer_count  total  refer_rate
true_cls_nm                                
CNV                  307   2246      0.1367
DME                   81    694      0.1167
DRUSEN               383    529      0.7240
NORMAL               549   1593      0.3446
  Saved → /kaggle/working/davis_net_analysis/triage/paper/refer_rate_by_class.png
  Saved → /kaggle/working/davis_net_analysis/triage/paper/trust_refer_accuracy_gap.png

══ All sections complete. ══
Results written to: /kaggle/working/davis_net_analysis


In [17]:
# ============================================================
# SECTION 12: FINAL INVENTORY PRINT
# Run this last to confirm all expected files exist.
# ============================================================

expected = {
    'PAPER': [
        f'{BASE}/explainability/paper/drusen_correct_gallery.png',
        f'{BASE}/explainability/paper/drusen_miscls_cnv_gallery.png',
        f'{BASE}/explainability/paper/drusen_miscls_normal_gallery.png',
        f'{BASE}/explainability/tables/xai_quant_summary_cbam2.csv',
        f'{BASE}/explainability/tables/cosine_sim_per_class_cbam2.csv',
        f'{BASE}/uncertainty/paper/boxplot_correct_vs_incorrect.png',
        f'{BASE}/uncertainty/paper/classwise_entropy_bar.png',
        f'{BASE}/uncertainty/paper/drusen_entropy_dist.png',
        f'{BASE}/uncertainty/tables/correct_vs_incorrect.csv',
        f'{BASE}/uncertainty/tables/significance_tests.csv',
        f'{BASE}/uncertainty/tables/classwise_uncertainty.csv',
        f'{BASE}/uncertainty/tables/drusen_roc_auc.csv',
        f'{BASE}/calibration/paper/reliability_diagram.png',
        f'{BASE}/calibration/tables/calibration_metrics.csv',
        f'{BASE}/conformal/paper/ambiguity_by_class_bar.png',
        f'{BASE}/conformal/tables/classwise_conformal_99pct.csv',
        f'{BASE}/conformal/tables/chi_square_ambiguity.csv',
        f'{BASE}/conformal/tables/drusen_pairing_counts.csv',
        f'{BASE}/joint/paper/correlation_heatmap.png',
        f'{BASE}/joint/tables/correlation_stats.csv',
        f'{BASE}/triage/paper/refer_rate_by_class.png',
        f'{BASE}/triage/paper/trust_refer_accuracy_gap.png',
        f'{BASE}/triage/tables/refer_rate_by_class.csv',
        f'{BASE}/triage/tables/trust_refer_summary.csv',
    ],
    'SUPPLEMENTARY / DIAGNOSTIC': [
        f'{BASE}/explainability/raw/quant_xai_per_image.csv',
        f'{BASE}/explainability/raw/drusen_localization_scores.csv',
        f'{BASE}/uncertainty/raw/mc_dropout_per_sample.csv',
        f'{BASE}/uncertainty/supp/full_metric_distributions.png',
        f'{BASE}/calibration/raw/test_probs_before_ts.npy',
        f'{BASE}/calibration/raw/test_probs_after_ts.npy',
        f'{BASE}/conformal/raw/conformal_per_sample_99pct.csv',
        f'{BASE}/joint/raw/joint_drusen_df.csv',
        f'{BASE}/joint/supp/scatter_pairs.png',
        f'{BASE}/triage/raw/per_sample_triage.csv',
    ],
}

for category, files in expected.items():
    missing = [f for f in files if not os.path.exists(f)]
    print(f"\n[{category}]  {len(files) - len(missing)}/{len(files)} present")
    for m in missing:
        print(f"  MISSING: {m}")


[PAPER]  24/24 present

[SUPPLEMENTARY / DIAGNOSTIC]  10/10 present


In [18]:
import shutil, os

shutil.make_archive(
    '/kaggle/working/davis_net_analysis_COMPLETE',
    'zip',
    '/kaggle/working/davis_net_analysis'
)
size_mb = os.path.getsize('/kaggle/working/davis_net_analysis_COMPLETE.zip') / 1e6
print(f"Saved: {size_mb:.1f} MB")

Saved: 5.2 MB


In [19]:
# ============================================================
# SECTION 12: BROADER (WHOLE-DATASET) ANALYSES
# Continuing in the same session — model, mc_df, conf_df,
# loc_df, quant_df, y_true_test, y_pred_test all still in memory.
# ============================================================
import os
os.makedirs(f"{BASE}/broader/paper", exist_ok=True)
os.makedirs(f"{BASE}/broader/tables", exist_ok=True)
os.makedirs(f"{BASE}/broader/raw", exist_ok=True)

# Recreate hooks (removed at end of Section 6)
hook_c1 = HookStore(model.cbam1)
hook_c2 = HookStore(model.cbam2)

In [20]:
# ── A. Selective Prediction Curve (whole dataset + per-class) ──
thresholds = np.linspace(mc_df['entropy'].min(), mc_df['entropy'].max(), 60)

def risk_coverage(df, thresholds):
    rows = []
    n_total = len(df)
    for t in thresholds:
        retained = df[df['entropy'] <= t]
        if len(retained) == 0:
            continue
        acc = (retained['correct'] == 1).mean()
        coverage = len(retained) / n_total
        rows.append({'threshold': t, 'coverage': coverage, 'accuracy': acc})
    return pd.DataFrame(rows)

# Pooled (whole dataset)
rc_pooled = risk_coverage(mc_df, thresholds)
rc_pooled.to_csv(f"{BASE}/broader/tables/risk_coverage_pooled.csv", index=False)

# Per-class
rc_per_class = {}
for cls in CLASSES:
    sub = mc_df[mc_df.true_cls_nm == cls]
    rc_per_class[cls] = risk_coverage(sub, thresholds)
    rc_per_class[cls].to_csv(f"{BASE}/broader/tables/risk_coverage_{cls}.csv", index=False)

# [PAPER] Figure
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(rc_pooled['coverage'], rc_pooled['accuracy'],
        color='black', linewidth=2.5, label='All Classes (Pooled)', zorder=5)
for cls in CLASSES:
    ax.plot(rc_per_class[cls]['coverage'], rc_per_class[cls]['accuracy'],
            color=CMAP[cls], linewidth=1.5, alpha=0.85, label=cls)
ax.set_xlabel('Coverage (fraction of samples retained)')
ax.set_ylabel('Accuracy on retained samples')
ax.set_title('Selective Prediction: Accuracy vs. Coverage\n(Entropy-based Rejection)')
ax.legend(loc='lower left')
ax.set_xlim(0, 1.02)
savefig(f"{BASE}/broader/paper/risk_coverage_curve.png")

  Saved → /kaggle/working/davis_net_analysis/broader/paper/risk_coverage_curve.png


In [21]:
# ── B. Grad-CAM localization score — FULL TEST SET ─────────────
# (previously DRUSEN-only; extending to all 4 classes)
all_loc_records = []
for gidx in tqdm(range(len(test_ds_flat)), desc='Full-set localization scores'):
    img_t, label, path = test_ds_flat[gidx]
    img_t = img_t.unsqueeze(0)
    cam   = gradcam(model, hook_c2, img_t, label, DEVICE)
    H, W  = cam.shape
    r0, r1 = H // 4, 3 * H // 4
    c0, c1 = W // 4, 3 * W // 4
    roi_energy   = cam[r0:r1, c0:c1].sum()
    total_energy = cam.sum() + 1e-8
    all_loc_records.append(dict(
        global_idx=gidx, label=label, cls=CLASSES[label],
        pred=y_pred_test[gidx],
        correct=int(y_pred_test[gidx] == label),
        loc_score=float(roi_energy / total_energy),
    ))

all_loc_df = pd.DataFrame(all_loc_records)
all_loc_df.to_csv(f"{BASE}/broader/raw/full_localization_scores.csv", index=False)
print(f"Full-set localization scores computed: {len(all_loc_df)}")

Full-set localization scores:   0%|          | 0/12650 [00:00<?, ?it/s]

Full-set localization scores computed: 12650


In [22]:
# ── All-class joint correlation (entropy × set_size × localization) ──
# Need conformal set sizes for ALL classes on the evaluation split (already in conf_df)
mc_all   = mc_df[['global_idx', 'entropy', 'correct', 'true_cls_nm']].copy()
conf_all = conf_df[['global_idx', 'set_size', 'is_multi', 'true_cls_nm']].copy()
loc_all  = all_loc_df[['global_idx', 'loc_score']].copy()

joint_all = (mc_all
             .merge(conf_all[['global_idx', 'set_size', 'is_multi']], on='global_idx', how='inner')
             .merge(loc_all, on='global_idx', how='inner'))

joint_all.to_csv(f"{BASE}/broader/raw/joint_all_classes.csv", index=False)
print(f"Joint all-class dataframe — N = {len(joint_all)}")

# Per-class correlation table [PAPER]
corr_rows = []
for cls in CLASSES:
    sub = joint_all[joint_all.true_cls_nm == cls]
    if len(sub) < 5:
        continue
    r_es, p_es = pearsonr(sub['entropy'], sub['set_size'])
    r_el, p_el = pearsonr(sub['entropy'], sub['loc_score'])
    r_sl, p_sl = pearsonr(sub['set_size'], sub['loc_score'])
    corr_rows.append(dict(
        Class=cls, N=len(sub),
        Entropy_vs_SetSize_r=r_es, Entropy_vs_SetSize_p=p_es,
        Entropy_vs_Loc_r=r_el, Entropy_vs_Loc_p=p_el,
        SetSize_vs_Loc_r=r_sl, SetSize_vs_Loc_p=p_sl,
    ))
allclass_corr_df = pd.DataFrame(corr_rows)
print("\n── Per-Class Correlation (Entropy / Set Size / Localization) [PAPER] ──")
print(allclass_corr_df.to_string(index=False, float_format='%.4f'))
allclass_corr_df.to_csv(f"{BASE}/broader/tables/per_class_correlation.csv", index=False)

# [PAPER] Heatmap grid — one Pearson r matrix per class
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, cls in zip(axes, CLASSES):
    sub = joint_all[joint_all.true_cls_nm == cls][['entropy', 'set_size', 'loc_score']]
    r_matrix = sub.corr(method='pearson')
    r_matrix.index = r_matrix.columns = ['Entropy', 'SetSize', 'Loc.']
    sns.heatmap(r_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, ax=ax, cbar=False, linewidths=0.5)
    ax.set_title(cls)
fig.suptitle('Entropy × Conformal Set Size × Localization — Per Class', y=1.05)
savefig(f"{BASE}/broader/paper/per_class_correlation_grid.png")

Joint all-class dataframe — N = 5062

── Per-Class Correlation (Entropy / Set Size / Localization) [PAPER] ──
 Class    N  Entropy_vs_SetSize_r  Entropy_vs_SetSize_p  Entropy_vs_Loc_r  Entropy_vs_Loc_p  SetSize_vs_Loc_r  SetSize_vs_Loc_p
   CNV 2246                0.6858                0.0000            0.0066            0.7548           -0.0145            0.4916
   DME  694                0.6676                0.0000           -0.0732            0.0540           -0.0284            0.4551
DRUSEN  529                0.7833                0.0000           -0.0601            0.1672           -0.0140            0.7486
NORMAL 1593                0.6856                0.0000           -0.0703            0.0050           -0.0333            0.1841
  Saved → /kaggle/working/davis_net_analysis/broader/paper/per_class_correlation_grid.png


In [23]:
# ── C. Confusion matrix annotated with mean entropy per cell ───
cm = confusion_matrix(y_true_test, y_pred_test)

entropy_by_cell = np.zeros((N_CLASSES, N_CLASSES))
for t in range(N_CLASSES):
    for p in range(N_CLASSES):
        mask = (mc_df['true_class'] == t) & (mc_df['pred_class'] == p)
        entropy_by_cell[t, p] = mc_df.loc[mask, 'entropy'].mean() if mask.sum() > 0 else np.nan

# [PAPER] Two-panel figure: counts | mean entropy
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASSES, yticklabels=CLASSES, cbar=False)
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(entropy_by_cell, annot=True, fmt='.3f', cmap='rocket_r', ax=axes[1],
            xticklabels=CLASSES, yticklabels=CLASSES,
            cbar_kws={'label': 'Mean Entropy'})
axes[1].set_title('Mean Predictive Entropy per Cell')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
savefig(f"{BASE}/broader/paper/confusion_matrix_entropy.png")

pd.DataFrame(entropy_by_cell, index=CLASSES, columns=CLASSES).to_csv(
    f"{BASE}/broader/tables/entropy_confusion_matrix.csv")

  Saved → /kaggle/working/davis_net_analysis/broader/paper/confusion_matrix_entropy.png


In [24]:
# ── D. Per-class ECE, before/after temperature scaling ─────────
cal_rows = []
for i, cls in enumerate(CLASSES):
    mask = test_labels_np == i
    ece_b = ece(probs_before[mask], test_labels_np[mask])
    ece_a = ece(probs_after[mask],  test_labels_np[mask])
    cal_rows.append(dict(Class=cls, ECE_Before_TS=ece_b, ECE_After_TS=ece_a,
                         Reduction_pct=(ece_b - ece_a) / ece_b * 100 if ece_b > 0 else np.nan))
perclass_cal_df = pd.DataFrame(cal_rows)
print("\n── Per-Class Calibration [PAPER] ──")
print(perclass_cal_df.to_string(index=False, float_format='%.4f'))
perclass_cal_df.to_csv(f"{BASE}/broader/tables/per_class_calibration.csv", index=False)


── Per-Class Calibration [PAPER] ──
 Class  ECE_Before_TS  ECE_After_TS  Reduction_pct
   CNV         0.0218        0.0083        61.8449
   DME         0.0100        0.0077        22.8341
DRUSEN         0.0617        0.0368        40.3562
NORMAL         0.0421        0.0135        68.0563


In [25]:
# ── E. Per-class ROAD / Avg Drop / Cosine Sim, CBAM-2 only ─────
rows = []
for mname in ['GradCAM', 'GradCAM++', 'ScoreCAM']:
    sub_m = quant_df[(quant_df.method == mname) & (quant_df.layer == 'CBAM2')]
    for cls in CLASSES:
        sub_c = sub_m[sub_m.cls == cls]
        cos   = cosine_sim_within_class(cam_store[mname]['CBAM2'][cls])
        rows.append(dict(
            Method=mname, Class=cls,
            ROAD=sub_c['road'].mean(),
            Average_Drop=sub_c['avg_drop'].mean(),
            Cosine_Sim=cos,
        ))
perclass_xai_df = pd.DataFrame(rows)
print("\n── Per-Class Quantitative XAI (CBAM-2) [PAPER] ──")
print(perclass_xai_df.to_string(index=False, float_format='%.4f'))
perclass_xai_df.to_csv(f"{BASE}/broader/tables/per_class_xai.csv", index=False)

# [PAPER] Grouped bar chart — ROAD by class × method
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=perclass_xai_df, x='Class', y='ROAD', hue='Method',
            order=CLASSES, ax=ax)
ax.set_title('ROAD Score by Class and XAI Method (CBAM-2)')
ax.set_ylabel('ROAD (higher = more faithful)')
savefig(f"{BASE}/broader/paper/road_by_class_method.png")


── Per-Class Quantitative XAI (CBAM-2) [PAPER] ──
   Method  Class    ROAD  Average_Drop  Cosine_Sim
  GradCAM    CNV 41.6220       42.6376      0.6803
  GradCAM    DME 47.9338       49.8185      0.7022
  GradCAM DRUSEN 76.4500       89.1208      0.7193
  GradCAM NORMAL 34.2520       37.5570      0.7637
GradCAM++    CNV 40.9063       41.9212      0.7001
GradCAM++    DME 49.0611       51.9535      0.7079
GradCAM++ DRUSEN 75.5622       86.8131      0.7415
GradCAM++ NORMAL 34.0191       36.8587      0.7652
 ScoreCAM    CNV 41.7981       42.8283      0.6865
 ScoreCAM    DME 48.6272       51.2731      0.6699
 ScoreCAM DRUSEN 75.7792       87.1470      0.7271
 ScoreCAM NORMAL 34.2852       37.5720      0.7621
  Saved → /kaggle/working/davis_net_analysis/broader/paper/road_by_class_method.png


In [31]:
# ── Recreate hooks fresh, right before Section 13 (Three-Way Split) ──
try:
    hook_c1.remove()
    hook_c2.remove()
except Exception:
    pass   # already removed or never attached — fine either way

hook_c1 = HookStore(model.cbam1)
hook_c2 = HookStore(model.cbam2)
print("Hooks reattached to cbam1 and cbam2.")

# Quick sanity check before running the full loop
_test_img, _test_label, _ = test_ds_flat[drusen_idx[0]]
_test_img = _test_img.unsqueeze(0)
_cam_check = gradcam(model, hook_c2, _test_img, _test_label, DEVICE)
assert _cam_check is not None and _cam_check.shape == (IMG_SIZE, IMG_SIZE), \
    "Hook still not firing — check model.cbam2 attribute path"
print("Hook sanity check passed. Proceeding with full three-way loop.")

Hooks reattached to cbam1 and cbam2.
Hook sanity check passed. Proceeding with full three-way loop.


In [32]:
# ============================================================
# SECTION 13: THREE-WAY DRUSEN OUTCOME SPLIT
# Correct | Misclassified→CNV | Misclassified→NORMAL
# ============================================================
os.makedirs(f"{BASE}/three_way/paper", exist_ok=True)
os.makedirs(f"{BASE}/three_way/tables", exist_ok=True)
os.makedirs(f"{BASE}/three_way/raw", exist_ok=True)

drusen_idx = np.where(y_true_test == 2)[0]   # DRUSEN = class 2

def outcome_group(gidx):
    pred = y_pred_test[gidx]
    if pred == 2:   return 'Correct'
    if pred == 0:   return 'Misclass_to_CNV'
    if pred == 3:   return 'Misclass_to_NORMAL'
    return 'Misclass_to_DME'   # rare, kept separate, not a focus group

# ── Compute heatmap centroid (center of mass) + spread ─────────
def cam_centroid(cam: np.ndarray):
    """Returns (row_frac, col_frac, dist_from_center) in [0,1]."""
    H, W = cam.shape
    ys, xs = np.mgrid[0:H, 0:W]
    total = cam.sum() + 1e-8
    cy = (ys * cam).sum() / total
    cx = (xs * cam).sum() / total
    row_frac = cy / H
    col_frac = cx / W
    dist_from_center = np.sqrt((row_frac - 0.5)**2 + (col_frac - 0.5)**2) / np.sqrt(0.5)
    return row_frac, col_frac, dist_from_center

three_way_records = []
for gidx in tqdm(drusen_idx, desc='Three-way DRUSEN analysis'):
    img_t, label, path = test_ds_flat[gidx]
    img_t = img_t.unsqueeze(0)
    cam   = gradcam(model, hook_c2, img_t, label, DEVICE)   # target = true DRUSEN class
    row_frac, col_frac, dist_ctr = cam_centroid(cam)

    grp = outcome_group(gidx)

    # Pull entropy/confidence from mc_df, loc_score from all_loc_df
    mc_row  = mc_df[mc_df.global_idx == gidx]
    loc_row = all_loc_df[all_loc_df.global_idx == gidx]

    three_way_records.append(dict(
        global_idx=gidx, group=grp,
        entropy=mc_row['entropy'].values[0] if len(mc_row) else np.nan,
        confidence=mc_row['confidence'].values[0] if len(mc_row) else np.nan,
        loc_score=loc_row['loc_score'].values[0] if len(loc_row) else np.nan,
        centroid_row=row_frac, centroid_col=col_frac,
        dist_from_center=dist_ctr,
    ))

three_way_df = pd.DataFrame(three_way_records)
three_way_df.to_csv(f"{BASE}/three_way/raw/three_way_per_sample.csv", index=False)
print(f"\nGroup sizes:\n{three_way_df['group'].value_counts()}")

# ── Summary table [PAPER] ───────────────────────────────────────
focus_groups = ['Correct', 'Misclass_to_CNV', 'Misclass_to_NORMAL']
tw_focus = three_way_df[three_way_df.group.isin(focus_groups)]

summary = tw_focus.groupby('group')[
    ['entropy', 'confidence', 'loc_score', 'dist_from_center']
].agg(['mean', 'std']).round(4)
summary = summary.reindex(focus_groups)
print("\n── Three-Way Outcome Summary [PAPER] ──")
print(summary.to_string())
summary.to_csv(f"{BASE}/three_way/tables/three_way_summary.csv")

# ── Statistical tests: Kruskal-Wallis across 3 groups, then pairwise ──
test_rows = []
for metric in ['entropy', 'confidence', 'loc_score', 'dist_from_center']:
    groups_data = [tw_focus[tw_focus.group == g][metric].dropna().values
                   for g in focus_groups]
    kw_stat, kw_p = stats.kruskal(*groups_data)
    test_rows.append(dict(Metric=metric, Test='Kruskal-Wallis (3-group)',
                          Statistic=kw_stat, P_Value=kw_p))

    # Pairwise Mann-Whitney (Bonferroni-adjusted for 3 comparisons)
    pairs = [('Correct', 'Misclass_to_CNV'),
             ('Correct', 'Misclass_to_NORMAL'),
             ('Misclass_to_CNV', 'Misclass_to_NORMAL')]
    for g1, g2 in pairs:
        a = tw_focus[tw_focus.group == g1][metric].dropna().values
        b = tw_focus[tw_focus.group == g2][metric].dropna().values
        if len(a) > 0 and len(b) > 0:
            u_stat, u_p = mannwhitneyu(a, b, alternative='two-sided')
            p_adj = min(u_p * 3, 1.0)
            test_rows.append(dict(Metric=metric, Test=f'{g1} vs {g2}',
                                  Statistic=u_stat, P_Value=p_adj))

tw_test_df = pd.DataFrame(test_rows)
print("\n── Statistical Tests [PAPER] ──")
print(tw_test_df.to_string(index=False, float_format='%.4e'))
tw_test_df.to_csv(f"{BASE}/three_way/tables/three_way_significance.csv", index=False)

# ── [PAPER] Boxplot comparison across 3 groups ──────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
metric_labels = {
    'entropy': 'Predictive Entropy',
    'confidence': 'Confidence',
    'loc_score': 'Localization Score',
    'dist_from_center': 'Heatmap Distance from Center',
}
group_colors = {'Correct': '#5B8DB8', 'Misclass_to_CNV': '#E07B54',
                'Misclass_to_NORMAL': '#8B7EC8'}
group_labels = {'Correct': 'Correct', 'Misclass_to_CNV': '→ CNV',
                'Misclass_to_NORMAL': '→ NORMAL'}

for ax, metric in zip(axes, metric_labels):
    plot_df = tw_focus.copy()
    plot_df['Group'] = plot_df['group'].map(group_labels)
    sns.boxplot(data=plot_df, x='Group', y=metric, ax=ax, width=0.55,
                order=['Correct', '→ CNV', '→ NORMAL'],
                palette={'Correct': '#5B8DB8', '→ CNV': '#E07B54', '→ NORMAL': '#8B7EC8'})
    ax.set_title(metric_labels[metric]); ax.set_xlabel('')
fig.suptitle('DRUSEN Three-Way Outcome Comparison', y=1.03, fontweight='bold')
savefig(f"{BASE}/three_way/paper/three_way_boxplots.png")

# ── [PAPER] Gallery: 3 examples per group, CBAM-2 Grad-CAM ──────
def make_three_way_gallery(n_per_group=3):
    fig, axes = plt.subplots(3, 3, figsize=(11, 11))
    for row, grp in enumerate(focus_groups):
        grp_idx = tw_focus[tw_focus.group == grp]['global_idx'].values
        chosen  = rng.choice(grp_idx, min(n_per_group, len(grp_idx)), replace=False)
        for col, gidx in enumerate(chosen[:3]):
            img_t, label, path = test_ds_flat[gidx]
            img_t  = img_t.unsqueeze(0)
            img_np = get_unnorm(img_t)
            cam    = gradcam(model, hook_c2, img_t, label, DEVICE)
            axes[row, col].imshow(overlay_cam(img_np, cam))
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(group_labels[grp], fontsize=13, fontweight='bold')
        axes[row, 0].axis('on')
        axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    fig.suptitle('DRUSEN — Grad-CAM by Outcome Group (CBAM-2)', fontweight='bold')
    savefig(f"{BASE}/three_way/paper/three_way_gallery.png")

make_three_way_gallery()

Three-way DRUSEN analysis:   0%|          | 0/1321 [00:00<?, ?it/s]


Group sizes:
group
Correct               1218
Misclass_to_NORMAL      49
Misclass_to_CNV         46
Misclass_to_DME          8
Name: count, dtype: int64

── Three-Way Outcome Summary [PAPER] ──
                   entropy         confidence         loc_score         dist_from_center        
                      mean     std       mean     std      mean     std             mean     std
group                                                                                           
Correct             0.3582  0.1726     0.8676  0.1034    0.6367  0.1170           0.1195  0.0715
Misclass_to_CNV     0.5404  0.1872     0.7465  0.1445    0.4860  0.1967           0.1427  0.0739
Misclass_to_NORMAL  0.5322  0.1996     0.7526  0.1546    0.5913  0.2617           0.2281  0.1858

── Statistical Tests [PAPER] ──
          Metric                                  Test  Statistic    P_Value
         entropy              Kruskal-Wallis (3-group) 6.8686e+01 1.2163e-15
         entropy            Correct 

In [34]:
# ============================================================
# SECTION 14: CONFORMAL PAIRING STRATIFICATION
# Singleton vs. (CNV,DRUSEN) vs. (DRUSEN,NORMAL)
# ============================================================
os.makedirs(f"{BASE}/pairing/paper", exist_ok=True)
os.makedirs(f"{BASE}/pairing/tables", exist_ok=True)

drusen_conf = conf_df[conf_df.true_cls_nm == 'DRUSEN'].copy()

def classify_pairing(pset):
    s = tuple(sorted(pset))
    if s == ('DRUSEN',):                return 'Singleton'
    if s == ('CNV', 'DRUSEN'):          return 'CNV_DRUSEN'
    if s == ('DRUSEN', 'NORMAL'):       return 'DRUSEN_NORMAL'
    return 'Other'   # e.g. (DME, DRUSEN) — rare, excluded from focus comparison

drusen_conf['pairing_group'] = drusen_conf['pred_set'].apply(classify_pairing)
print(drusen_conf['pairing_group'].value_counts())

# ── Merge with entropy (mc_df) and localization (all_loc_df) ───
pairing_df = (drusen_conf
              .merge(mc_df[['global_idx', 'entropy', 'confidence']], on='global_idx', how='left')
              .merge(all_loc_df[['global_idx', 'loc_score']], on='global_idx', how='left'))

pairing_df.to_csv(f"{BASE}/pairing/tables/pairing_per_sample.csv", index=False)

focus_pairs = ['Singleton', 'CNV_DRUSEN', 'DRUSEN_NORMAL']
pf = pairing_df[pairing_df.pairing_group.isin(focus_pairs)]

# ── Summary table [PAPER] ───────────────────────────────────────
pairing_summary = pf.groupby('pairing_group')[['entropy', 'confidence', 'loc_score']] \
                    .agg(['mean', 'std', 'count']).round(4).reindex(focus_pairs)
print("\n── Conformal Pairing Summary [PAPER] ──")
print(pairing_summary.to_string())
pairing_summary.to_csv(f"{BASE}/pairing/tables/pairing_summary.csv")

# ── Statistical tests: does entropy distinguish the two rival types? ──
# Key test: CNV_DRUSEN vs. DRUSEN_NORMAL directly (not just each vs. Singleton)
test_rows = []
for metric in ['entropy', 'confidence', 'loc_score']:
    groups_data = [pf[pf.pairing_group == g][metric].dropna().values for g in focus_pairs]
    if all(len(g) > 0 for g in groups_data):
        kw_stat, kw_p = stats.kruskal(*groups_data)
        test_rows.append(dict(Metric=metric, Test='Kruskal-Wallis (3-group)',
                              Statistic=kw_stat, P_Value=kw_p))

    pairs = [('Singleton', 'CNV_DRUSEN'),
             ('Singleton', 'DRUSEN_NORMAL'),
             ('CNV_DRUSEN', 'DRUSEN_NORMAL')]     # ← the critical comparison
    for g1, g2 in pairs:
        a = pf[pf.pairing_group == g1][metric].dropna().values
        b = pf[pf.pairing_group == g2][metric].dropna().values
        if len(a) > 0 and len(b) > 0:
            u_stat, u_p = mannwhitneyu(a, b, alternative='two-sided')
            p_adj = min(u_p * 3, 1.0)
            test_rows.append(dict(Metric=metric, Test=f'{g1} vs {g2}',
                                  Statistic=u_stat, P_Value=p_adj))

pairing_test_df = pd.DataFrame(test_rows)
print("\n── Pairing Statistical Tests [PAPER] ──")
print(pairing_test_df.to_string(index=False, float_format='%.4e'))
pairing_test_df.to_csv(f"{BASE}/pairing/tables/pairing_significance.csv", index=False)

# Highlight the key result
key_row = pairing_test_df[pairing_test_df.Test == 'CNV_DRUSEN vs DRUSEN_NORMAL']
print("\n>>> KEY RESULT — does entropy separate the two rival types? <<<")
print(key_row.to_string(index=False, float_format='%.4e'))

# ── [PAPER] Boxplot: entropy and localization across 3 groups ──
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_labels = {'Singleton': 'Singleton\n(Confident)', 'CNV_DRUSEN': '(CNV,\nDRUSEN)',
              'DRUSEN_NORMAL': '(DRUSEN,\nNORMAL)'}
pf_plot = pf.copy()
pf_plot['Group'] = pf_plot['pairing_group'].map(plot_labels)
order = [plot_labels[g] for g in focus_pairs]

for ax, metric, ylab in zip(axes, ['entropy', 'loc_score'],
                             ['Predictive Entropy', 'Localization Score']):
    sns.boxplot(data=pf_plot, x='Group', y=metric, ax=ax, order=order, width=0.55,
                palette=['#5B8DB8', '#E07B54', '#8B7EC8'])
    ax.set_title(ylab); ax.set_xlabel('')
fig.suptitle('DRUSEN Conformal Pairing — Entropy & Localization by Ambiguity Type',
            y=1.03, fontweight='bold')
savefig(f"{BASE}/pairing/paper/pairing_boxplots.png")

pairing_group
Singleton        384
CNV_DRUSEN        84
DRUSEN_NORMAL     43
Other             18
Name: count, dtype: int64

── Conformal Pairing Summary [PAPER] ──
              entropy               confidence               loc_score              
                 mean     std count       mean     std count      mean     std count
pairing_group                                                                       
Singleton      0.2906  0.1190   384     0.9129  0.0493   384    0.6439  0.1139   384
CNV_DRUSEN     0.6060  0.0627    84     0.7028  0.0788    84    0.6306  0.1099    84
DRUSEN_NORMAL  0.6451  0.0682    43     0.6750  0.0842    43    0.6639  0.0988    43

── Pairing Statistical Tests [PAPER] ──
    Metric                        Test  Statistic    P_Value
   entropy    Kruskal-Wallis (3-group) 2.8401e+02 2.1231e-62
   entropy     Singleton vs CNV_DRUSEN 8.0000e+01 7.3175e-46
   entropy  Singleton vs DRUSEN_NORMAL 2.0000e+01 2.1732e-26
   entropy CNV_DRUSEN vs DRUSEN_NORMAL 1

In [28]:
# ── Cleanup and save ─────────────────────────────────────────
hook_c1.remove()
hook_c2.remove()

import shutil
shutil.make_archive(
    '/kaggle/working/davis_net_analysis_EXTENDED',
    'zip',
    '/kaggle/working/davis_net_analysis'
)
print("Extended analysis complete and zipped.")

Extended analysis complete and zipped.


In [35]:
# ── Diagnostic: is "road" actually just "avg_drop" in disguise? ──
corr = quant_df[quant_df.layer == 'CBAM2'][['road', 'avg_drop']].corr()
print(corr)

# Also check per-class
for cls in CLASSES:
    sub = quant_df[(quant_df.layer == 'CBAM2') & (quant_df.cls == cls)]
    r = sub['road'].corr(sub['avg_drop'])
    print(f"{cls}: r(road, avg_drop) = {r:.4f}")

              road  avg_drop
road      1.000000  0.963064
avg_drop  0.963064  1.000000
CNV: r(road, avg_drop) = 0.9987
DME: r(road, avg_drop) = 0.9689
DRUSEN: r(road, avg_drop) = 0.8255
NORMAL: r(road, avg_drop) = 0.9464


In [36]:
# ── Rename ROAD → Confidence_Drop_Top20pct across all saved tables ──

# 1. Main summary table (Section 6.5)
xai_table = xai_table.rename(columns={'ROAD': 'Confidence_Drop_Top20pct'})
xai_table.to_csv(f"{BASE}/explainability/tables/xai_quant_summary_cbam2.csv", index=False)

# 2. Per-class breakdown (Section 12E)
perclass_xai_df = perclass_xai_df.rename(columns={'ROAD': 'Confidence_Drop_Top20pct'})
perclass_xai_df.to_csv(f"{BASE}/broader/tables/per_class_xai.csv", index=False)

# 3. Raw per-image records — rename column, keep everything else
quant_df = quant_df.rename(columns={'road': 'confidence_drop_top20pct'})
quant_df.to_csv(f"{BASE}/explainability/raw/quant_xai_per_image.csv", index=False)

print("Renamed. Re-check tables:")
print(xai_table)
print(perclass_xai_df)

Renamed. Re-check tables:
      Method  Confidence_Drop_Top20pct  Average_Drop  Incr_Conf_pct  \
0    GradCAM                 50.064448     54.783455            3.0   
1  GradCAM++                 49.887180     54.386651            3.0   
2   ScoreCAM                 50.122448     54.705078            2.5   

   Cosine_Sim  
0    0.716380  
1    0.728680  
2    0.711415  
       Method   Class  Confidence_Drop_Top20pct  Average_Drop  Cosine_Sim
0     GradCAM     CNV                 41.622004     42.637624    0.680299
1     GradCAM     DME                 47.933767     49.818457    0.702168
2     GradCAM  DRUSEN                 76.450004     89.120776    0.719308
3     GradCAM  NORMAL                 34.252019     37.556963    0.763744
4   GradCAM++     CNV                 40.906261     41.921234    0.700082
5   GradCAM++     DME                 49.061105     51.953525    0.707926
6   GradCAM++  DRUSEN                 75.562243     86.813128    0.741537
7   GradCAM++  NORMAL            

In [37]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=perclass_xai_df, x='Class', y='Confidence_Drop_Top20pct', hue='Method',
            order=CLASSES, ax=ax)
ax.set_title('Confidence Drop (Top-20% Masked Pixels) by Class and XAI Method')
ax.set_ylabel('Confidence Drop (%)  —  higher = more concentrated attention')
savefig(f"{BASE}/broader/paper/confdrop_by_class_method.png")

  Saved → /kaggle/working/davis_net_analysis/broader/paper/confdrop_by_class_method.png


In [38]:
import shutil
shutil.make_archive(
    '/kaggle/working/davis_net_analysis_FINAL',
    'zip',
    '/kaggle/working/davis_net_analysis'
)
print("Saved.")

Saved.
